# 02 - Scrub: Pembersihan dan Integrasi Data

**Dikerjakan oleh:** Ghazy Achmed Movlech Urbayani (18223093) - Data Preprocessing Lead

**Kelompok 14 | II4013 Data Analitik**

Notebook ini adalah tahap kedua dari kerangka OSEMN. Tujuannya adalah membersihkan kedua dataset dari masalah kualitas data seperti nilai kosong, duplikasi, outlier, dan format yang tidak konsisten, kemudian membuat fitur-fitur baru yang dibutuhkan untuk analisis dan pemodelan. Setiap keputusan pembersihan dicatat beserta alasannya agar proses ini dapat direproduksi dan diverifikasi.

## 1. Persiapan: Import Library dan Muat Data

Pada bagian ini kita menyiapkan semua library yang dibutuhkan dan memuat semua file dataset mentah ke dalam memori. Selain dataset utama, terdapat juga beberapa file pendukung dari DS2 yang saling berhubungan.

In [1]:
# Setup library dan path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 65)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

DS1_PATH       = '../data/raw/primary/WA_Fn-UseC_-IT-Help-Desk.csv'
DS2_ISSUES     = '../data/raw/supporting/issues.csv'
DS2_HISTORY    = '../data/raw/supporting/issues_change_history.csv'
DS2_SNAPSHOT   = '../data/raw/supporting/issues_snapshot.csv'
DS2_SCORED     = '../data/raw/supporting/issues_snapshot_sample.xlsx'
DS2_UTTERANCES = '../data/raw/supporting/sample_utterances.csv'
OUTPUT_PATH    = '../data/processed/'

os.makedirs(OUTPUT_PATH, exist_ok=True)
print('Setup selesai')

Setup selesai


In [2]:
# Load DS1 (Primary — IBM Watson IT Help Desk)
df1 = pd.read_csv(DS1_PATH)
print('DS1 shape:', df1.shape)
print('Kolom DS1:', list(df1.columns))

DS1 shape: (100000, 10)
Kolom DS1: ['ticket', 'requestor', 'RequestorSeniority', 'ITOwner', 'FiledAgainst', 'TicketType', 'Severity', 'Priority', 'daysOpen', 'Satisfaction']


In [3]:
# DS1 Info & Head
df1.info()
df1.head(5)

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   ticket              100000 non-null  int64
 1   requestor           100000 non-null  int64
 2   RequestorSeniority  100000 non-null  str  
 3   ITOwner             100000 non-null  int64
 4   FiledAgainst        100000 non-null  str  
 5   TicketType          100000 non-null  str  
 6   Severity            100000 non-null  str  
 7   Priority            100000 non-null  str  
 8   daysOpen            100000 non-null  int64
 9   Satisfaction        100000 non-null  str  
dtypes: int64(4), str(6)
memory usage: 7.6 MB


,ticket,requestor,RequestorSeniority,ITOwner,FiledAgainst,TicketType,Severity,Priority,daysOpen,Satisfaction
0,1,1929,1 - Junior,50,Systems,Issue,2 - Normal,0 - Unassigned,3,1 - Unsatisfied
1,2,1587,2 - Regular,15,Software,Request,1 - Minor,1 - Low,5,1 - Unsatisfied
2,3,925,2 - Regular,15,Access/Login,Request,2 - Normal,0 - Unassigned,0,0 - Unknown
3,4,413,4 - Management,22,Systems,Request,2 - Normal,0 - Unassigned,20,0 - Unknown
4,5,318,1 - Junior,22,Access/Login,Request,2 - Normal,1 - Low,1,1 - Unsatisfied


In [4]:
# Load DS2 — semua file supporting
df_issues     = pd.read_csv(DS2_ISSUES, low_memory=False)
df_history    = pd.read_csv(DS2_HISTORY, low_memory=False)
df_snapshot   = pd.read_csv(DS2_SNAPSHOT, low_memory=False)
df_scored     = pd.read_excel(DS2_SCORED)
df_utterances = pd.read_csv(DS2_UTTERANCES, low_memory=False)
print('DS2 files loaded.')

DS2 files loaded.


In [5]:
# Ringkasan Shape Semua File
datasets = [
    ('WA_Fn-UseC_-IT-Help-Desk.csv  [DS1 primary]',  df1),
    ('issues.csv                    [DS2 backbone]',  df_issues),
    ('issues_change_history.csv     [DS2 history]',   df_history),
    ('issues_snapshot.csv           [DS2 per-turn]',  df_snapshot),
    ('issues_snapshot_sample.xlsx   [DS2 scored]',    df_scored),
    ('sample_utterances.csv         [DS2 teks]',      df_utterances),
]
print('{:<50} {:>9} {:>7}'.format('File', 'Baris', 'Kolom'))
print('-' * 68)
for name, df in datasets:
    print('{:<50} {:>9,} {:>7}'.format(name, df.shape[0], df.shape[1]))

File                                                   Baris   Kolom
--------------------------------------------------------------------
WA_Fn-UseC_-IT-Help-Desk.csv  [DS1 primary]          100,000      10
issues.csv                    [DS2 backbone]          66,691      58
issues_change_history.csv     [DS2 history]          257,508       6
issues_snapshot.csv           [DS2 per-turn]          90,963      60
issues_snapshot_sample.xlsx   [DS2 scored]               747      19
sample_utterances.csv         [DS2 teks]              30,104       9


In [ ]:
# Preview DS2 — head(3) tiap file
for label, df in [
    ('issues.csv', df_issues),
    ('issues_change_history.csv', df_history),
    ('issues_snapshot.csv', df_snapshot),
    ('issues_snapshot_sample.xlsx', df_scored),
    ('sample_utterances.csv', df_utterances),
]:
    print('\n=== ' + label + ' ===')
    display(df.head(3))


=== issues.csv ===


,id,started,ended,issue_num,issue_proj,issue_reporter,issue_assignee,issue_contr_count,issue_type,issue_priority,issue_created,issue_resolution_date,issue_resolution,issue_status,issue_comments_count,last_change_date,wf_in_review,wfe_in_review,wf_deployment,wfe_deployment,wf_resolved,wfe_resolved,wf_open,wfe_open,wf_monitoring,wfe_monitoring,wf_done,wfe_done,wf_pending_customer_approval,wfe_pending_customer_approval,wf_rejected,wfe_rejected,wf_testing_monitoring,wfe_testing_monitoring,wf_in_progress,wfe_in_progress,wf_reopened,wfe_reopened,wf_to_do,wfe_to_do,wf_validation,wfe_validation,wf_resolved_under_monitoring,wfe_resolved_under_monitoring,wf_closed,wfe_closed,wf_waiting,wfe_waiting,wf_cancelled,wfe_cancelled,wf_under_review,wfe_under_review,wf_approved,wfe_approved,wf_pending_deployment,wfe_pending_deployment,wf_total_time,processing_steps
0,11887.00,2016-01-06 08:23:43+00:00,2016-01-06 08:56:55+00:00,186.00,d1z0,4olg,NaN,1.00,Ticket,Medium,2016-01-06 08:23:43+00:00,2016-01-06 08:56:55+00:00,Done,done,1,2016-04-02 12:20:21+00:00,NaN,0,NaN,0,NaN,0,29.00,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,1963.00,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,1992.00,2
1,11890.00,2016-01-11 10:06:19+00:00,2016-01-12 12:30:23+00:00,190.00,d1z0,4olg,NaN,1.00,Ticket,Medium,2016-01-11 10:06:19+00:00,2016-01-12 12:30:23+00:00,Done,done,0,2016-04-02 12:20:21+00:00,NaN,0,NaN,0,NaN,0,44.00,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,95000.00,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,95044.00,2
2,11904.00,2016-01-21 07:28:20+00:00,2016-01-26 08:21:47+00:00,198.00,d1z0,4ohk,4ohk,1.00,Ticket,Medium,2016-01-21 07:28:20+00:00,2016-01-26 08:21:47+00:00,Done,done,0,2016-04-02 12:20:21+00:00,NaN,0,NaN,0,NaN,0,8.00,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,435199.00,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,435207.00,2



=== issues_change_history.csv ===


,id,issueid,field,value,created,change_group_id
0,10810.00,47751000.00,status,resolved,2016-03-24 15:35:53+00:00,10707.00
1,10821.00,47751000.00,status,resolved,2016-03-24 16:12:16+00:00,10715.00
2,10823.00,47751000.00,status,reopened,2016-03-24 16:12:19+00:00,10716.00



=== issues_snapshot.csv ===


,idx,id,started,ended,issue_num,issue_proj,issue_reporter,issue_assignee,issue_contr_count,issue_type,issue_priority,issue_created,issue_resolution_date,issue_resolution,issue_status,issue_comments_count,last_change_date,wf_in_review,wfe_in_review,wf_deployment,wfe_deployment,wf_resolved,wfe_resolved,wf_open,wfe_open,wf_monitoring,wfe_monitoring,wf_done,wfe_done,wf_pending_customer_approval,wfe_pending_customer_approval,wf_rejected,wfe_rejected,wf_testing_monitoring,wfe_testing_monitoring,wf_in_progress,wfe_in_progress,wf_reopened,wfe_reopened,wf_to_do,wfe_to_do,wf_validation,wfe_validation,wf_resolved_under_monitoring,wfe_resolved_under_monitoring,wf_closed,wfe_closed,wf_waiting,wfe_waiting,wf_cancelled,wfe_cancelled,wf_under_review,wfe_under_review,wf_approved,wfe_approved,wf_pending_deployment,wfe_pending_deployment,turn,wf_total_time,processing_steps
0,0,11887.00,2016-01-06 08:23:43+00:00,2016-01-06 08:56:55+00:00,186.00,d1z0,4olg,NaN,1.00,Ticket,Medium,2016-01-06 08:23:43+00:00,2016-01-06 08:56:55+00:00,Done,done,1,2016-04-02 12:20:21+00:00,NaN,0,NaN,0,NaN,0,29.00,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,1963.00,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,1,1992.00,2
1,1,11890.00,2016-01-11 10:06:19+00:00,2016-01-12 12:30:23+00:00,190.00,d1z0,4olg,NaN,1.00,Ticket,Medium,2016-01-11 10:06:19+00:00,2016-01-12 12:30:23+00:00,Done,done,0,2016-04-02 12:20:21+00:00,NaN,0,NaN,0,NaN,0,44.00,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,95000.00,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,1,95044.00,2
2,2,11904.00,2016-01-21 07:28:20+00:00,2016-01-26 08:21:47+00:00,198.00,d1z0,4ohk,4ohk,1.00,Ticket,Medium,2016-01-21 07:28:20+00:00,2016-01-26 08:21:47+00:00,Done,done,0,2016-04-02 12:20:21+00:00,NaN,0,NaN,0,NaN,0,8.00,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,435199.00,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,1,435207.00,2



=== issues_snapshot_sample.xlsx ===


,id,no,project,reporter,type,priority,contributors,turn_no,assignee,started,ended,spent hours,steps,comments count,valid,Q1,Q2,Q3,Notes
0,1004298.00,346.00,C11gg0w13,.wy0w13yhr,Ticket,High,1.00,1,4ops,2022-01-02 11:12:51+00:00,2022-05-24 05:40:13+00:00,3402.46,2,49,True,5,5,5,This is an HD-Service
1,1004357.00,155.00,C01gg1_3_,z8y1_3_ygi,Ticket,Medium,1.00,1,4lip,2022-01-05 05:27:14+00:00,2022-01-12 07:47:47+00:00,170.34,3,4,True,3,4,4,This is an HD-Service and looks like communica...
2,1004361.00,14.00,C02hk0x93,0ny0x93yik,Ticket,Medium,3.00,1,4hghs,2022-01-05 06:50:56+00:00,2022-01-05 06:55:23+00:00,0.07,2,3,True,5,5,5,Service Desk user did his work perfectly same ...



=== sample_utterances.csv ===


,issueid,id,comment_seq,utr_seq,author,created,is_private,actionbody,author_role
0,1004298.00,211718014.00,0,0,.wy0w13yhr,2022-01-02 11:12:51+00:00,0.00,dear ph_name team,reporter
1,1004298.00,211718014.00,0,1,.wy0w13yhr,2022-01-02 11:12:51+00:00,0.00,we need your urgent support to fix list of vul...,reporter
2,1004298.00,211718014.00,0,2,.wy0w13yhr,2022-01-02 11:12:51+00:00,0.00,please provide resolution date before end of ‘...,reporter


In [7]:
# Immutable Copies
df1_orig           = df1.copy()
df_issues_orig     = df_issues.copy()
df_history_orig    = df_history.copy()
df_snapshot_orig   = df_snapshot.copy()
df_scored_orig     = df_scored.copy()
df_utterances_orig = df_utterances.copy()
print('Copy immutable Saved.')

Copy immutable Saved.


In [ ]:
# Tabel Baseline — ringkasan semua file
baseline_info = [
    ('WA_Fn-UseC_-IT-Help-Desk.csv', df1.shape[0], df1.shape[1],
     'IBM Watson simulation; daysOpen=proxy durasi; 0 null'),
    ('issues.csv', df_issues.shape[0], df_issues.shape[1],
     'Real helpdesk Jan 2016-Mar 2023; wf_* missing by design'),
    ('issues_change_history.csv', df_history.shape[0], df_history.shape[1],
     'Histori perubahan assignee dan status workflow'),
    ('issues_snapshot.csv', df_snapshot.shape[0], df_snapshot.shape[1],
     'Per-assignee per-turn; baris > issues.csv adalah desain'),
    ('issues_snapshot_sample.xlsx', df_scored.shape[0], df_scored.shape[1],
     'Ground truth Q1/Q2/Q3 score 1-5; stratified sample'),
    ('sample_utterances.csv', df_utterances.shape[0], df_utterances.shape[1],
     'Teks percakapan; hanya tiket di scored sample'),
]
print('| {:<42} | {:>7} | {:>5} | {:<48} |'.format(
    'File', 'Baris', 'Kol', 'Catatan Awal'))
print('|' + '-'*44 + '|' + '-'*9 + '|' + '-'*7 + '|' + '-'*50 + '|')
for nm, r, c, note in baseline_info:
    print('| {:<42} | {:>7,} | {:>5} | {:<48} |'.format(nm, r, c, note))

| File                                       |   Baris |   Kol | Catatan Awal                                     |
|--------------------------------------------|---------|-------|--------------------------------------------------|
| WA_Fn-UseC_-IT-Help-Desk.csv               | 100,000 |    10 | IBM Watson simulation; daysOpen=proxy durasi; 0 null |
| issues.csv                                 |  66,691 |    58 | Real helpdesk Jan 2016-Mar 2023; wf_* missing by design |
| issues_change_history.csv                  | 257,508 |     6 | Histori perubahan assignee dan status workflow   |
| issues_snapshot.csv                        |  90,963 |    60 | Per-assignee per-turn; baris > issues.csv adalah desain |
| issues_snapshot_sample.xlsx                |     747 |    19 | Ground truth Q1/Q2/Q3 score 1-5; stratified sample |
| sample_utterances.csv                      |  30,104 |     9 | Teks percakapan; hanya tiket di scored sample    |


## 2. Analisis Data Eksploratif (Diagnostik)

Sebelum membersihkan data, kita perlu memahami kondisi aslinya terlebih dahulu. Bagian ini menjalankan berbagai pemeriksaan diagnostik untuk mengidentifikasi masalah yang ada di setiap dataset. Setiap keputusan pembersihan di bagian selanjutnya akan merujuk pada temuan dari bagian ini.

### 2.1 Distribusi Nilai dan Statistik Dasar

Kita melihat distribusi nilai di setiap kolom kategorikal dan statistik ringkasan (rata-rata, minimum, maksimum) untuk kolom numerik. Tujuannya adalah memahami karakteristik data sebelum diproses.

In [9]:
# DS1 — Value Counts + Persen Kolom Kategorikal

CAT_COLS_DS1 = ['RequestorSeniority', 'FiledAgainst', 'TicketType',
                'Severity', 'Priority', 'Satisfaction']

for col in CAT_COLS_DS1:
    vc  = df1[col].value_counts(dropna=False)
    pct = (vc / len(df1) * 100).round(2)
    tbl = pd.DataFrame({'count': vc, 'pct_%': pct})
    print('\n' + '='*52)
    print('  DS1 | ' + col + '  (n=' + str(len(df1)) + ')')
    print('='*52)
    print(tbl.to_string())


  DS1 | RequestorSeniority  (n=100000)
                    count  pct_%
RequestorSeniority              
2 - Regular         41303  41.30
1 - Junior          20040  20.04
4 - Management      19856  19.86
3 - Senior          18801  18.80

  DS1 | FiledAgainst  (n=100000)
              count  pct_%
FiledAgainst              
Systems       40035  40.03
Access/Login  29921  29.92
Software      20068  20.07
Hardware       9976   9.98

  DS1 | TicketType  (n=100000)
            count  pct_%
TicketType              
Request     75074  75.07
Issue       24926  24.93

  DS1 | Severity  (n=100000)
                  count  pct_%
Severity                      
2 - Normal        90912  90.91
3 - Major          4974   4.97
1 - Minor          2317   2.32
4 - Critical       1430   1.43
0 - Unclassified    367   0.37

  DS1 | Priority  (n=100000)
                count  pct_%
Priority                    
3 - High        36498  36.50
0 - Unassigned  30127  30.13
1 - Low         17117  17.12
2 - Medium  

In [10]:
# DS1 — Numeric Describe
print('=== DS1: daysOpen ===')
print(df1['daysOpen'].describe().round(2))
print()
print('Catatan: DS1 tidak punya reassignmentCount/reopenCount/sysModCount.')
print('(Kolom-kolom tersebut hanya ada di UCI Incident Log, bukan IBM Watson.)')
print('daysOpen adalah satu-satunya kolom numerik substantif di DS1.')

=== DS1: daysOpen ===
count   100000.00
mean         6.84
std          7.38
min          0.00
25%          1.00
50%          5.00
75%         10.00
max         54.00
Name: daysOpen, dtype: float64

Catatan: DS1 tidak punya reassignmentCount/reopenCount/sysModCount.
(Kolom-kolom tersebut hanya ada di UCI Incident Log, bukan IBM Watson.)
daysOpen adalah satu-satunya kolom numerik substantif di DS1.


In [11]:
# DS1 — Cross-check Priority vs Severity
# catatan: DS1 tidak punya auto-derived priority dari urgency x impact seperti UCI.
# cross-tab "untuk melihat "apakah ada pola konsisten Severity -> Priority"

ct = pd.crosstab(df1['Severity'], df1['Priority'])
print('Cross-tabulation DS1: Severity x Priority')
display(ct)

ct_pct = pd.crosstab(df1['Severity'], df1['Priority'], normalize='index').round(3) * 100
print('\nDistribusi % per baris (normalize=index):')
display(ct_pct)

print('\nTemuan: apakah Severity rendah -> Priority rendah secara konsisten?')
print('Jika ya, Priority bisa redundan dengan Severity (seperti priority auto-calc di UCI).')

Cross-tabulation DS1: Severity x Priority


Priority,0 - Unassigned,1 - Low,2 - Medium,3 - High
Severity,,,,
0 - Unclassified,120,82,57,108
1 - Minor,642,564,420,691
2 - Normal,27478,15670,14834,32930
3 - Major,1469,626,740,2139
4 - Critical,418,175,207,630



Distribusi % per baris (normalize=index):


Priority,0 - Unassigned,1 - Low,2 - Medium,3 - High
Severity,,,,
0 - Unclassified,32.70,22.30,15.50,29.40
1 - Minor,27.70,24.30,18.10,29.80
2 - Normal,30.20,17.20,16.30,36.20
3 - Major,29.50,12.60,14.90,43.00
4 - Critical,29.20,12.20,14.50,44.10



Temuan: apakah Severity rendah -> Priority rendah secara konsisten?
Jika ya, Priority bisa redundan dengan Severity (seperti priority auto-calc di UCI).


In [12]:
# DS2 issues — Value Counts + Persen Kolom Kategorikal

CAT_COLS_DS2 = ['issue_priority', 'issue_type', 'issue_status', 'issue_resolution']

for col in CAT_COLS_DS2:
    vc  = df_issues[col].value_counts(dropna=False)
    pct = (vc / len(df_issues) * 100).round(2)
    tbl = pd.DataFrame({'count': vc, 'pct_%': pct})
    print('\n' + '='*52)
    print('  DS2 issues | ' + col + '  (n=' + str(len(df_issues)) + ')')
    print('='*52)
    print(tbl.to_string())


  DS2 issues | issue_priority  (n=66691)
                count  pct_%
issue_priority              
unknown         33965  50.93
Medium          24788  37.17
High             4554   6.83
Highest          2084   3.12
Blocker           656   0.98
Low               560   0.84
Lowest             84   0.13

  DS2 issues | issue_type  (n=66691)
                count  pct_%
issue_type                  
Ticket          45275  67.89
Service          5300   7.95
Subtask          4746   7.12
Story            4538   6.80
HD Service       1686   2.53
Task             1540   2.31
Vacation          856   1.28
Project           842   1.26
Sub-task          544   0.82
Epic              403   0.60
Deployment        350   0.52
Retrospective     241   0.36
Sprint Summary    208   0.31
Assistance        109   0.16
Bug                53   0.08

  DS2 issues | issue_status  (n=66691)
                           count  pct_%
issue_status                           
closed                     56344  84.49
done  

In [13]:
# DS2 issues — Numeric Describe
NUM_COLS_DS2 = ['wf_total_time', 'processing_steps', 'issue_comments_count', 'issue_contr_count']
print('=== DS2 issues — kolom numerik utama ===')
display(df_issues[NUM_COLS_DS2].describe().round(2))

wt_median = df_issues['wf_total_time'].median()
wt_max    = df_issues['wf_total_time'].max()
print('\nwf_total_time satuan: DETIK (per FEATURES.md)')
print('Median: {:.0f} detik = {:.1f} jam'.format(wt_median, wt_median/3600))
print('Max   : {:.0f} detik = {:.1f} jam = {:.1f} hari'.format(wt_max, wt_max/3600, wt_max/86400))

=== DS2 issues — kolom numerik utama ===


,wf_total_time,processing_steps,issue_comments_count,issue_contr_count
count,66691.00,66691.00,66691.00,66691.00
mean,65856465.21,3.21,8.64,1.22
std,99715344.06,2.86,13.79,0.53
min,0.00,0.00,0.00,1.00
25%,440845.37,1.00,3.00,1.00
50%,2961971.00,3.00,5.00,1.00
75%,129318977.04,4.00,10.00,1.00
max,357193929.88,93.00,685.00,10.00



wf_total_time satuan: DETIK (per FEATURES.md)
Median: 2961971 detik = 822.8 jam
Max   : 357193930 detik = 99220.5 jam = 4134.2 hari


In [14]:
# [AI-assisted] Scored Sample — Struktur & Distribusi Q1/Q2/Q3
print('Scored sample shape:', df_scored.shape)
print('Kolom:', list(df_scored.columns))
print()

for q in ['Q1', 'Q2', 'Q3']:
    if q not in df_scored.columns:
        print(q + ' tidak ditemukan di scored sample!')
        continue
    vc  = df_scored[q].value_counts(dropna=False).sort_index()
    pct = (vc / vc.sum() * 100).round(2)
    tbl = pd.DataFrame({'count': vc, 'pct_%': pct})
    print('=== ' + q + ' (skala 1-5) ===')
    print(tbl.to_string())
    print()

if 'valid' in df_scored.columns:
    print('=== valid flag ===')
    print(df_scored['valid'].value_counts(dropna=False))

Scored sample shape: (747, 19)
Kolom: ['id', 'no', 'project', 'reporter', 'type', 'priority', 'contributors', 'turn_no', 'assignee', 'started', 'ended', 'spent hours', 'steps', 'comments count', 'valid', 'Q1', 'Q2', 'Q3', 'Notes']

=== Q1 (skala 1-5) ===
    count  pct_%
Q1              
0     144  19.28
1      50   6.69
2      42   5.62
3      67   8.97
4      72   9.64
5     372  49.80

=== Q2 (skala 1-5) ===
    count  pct_%
Q2              
0     144  19.28
1      51   6.83
2      42   5.62
3      72   9.64
4      70   9.37
5     368  49.26

=== Q3 (skala 1-5) ===
    count  pct_%
Q3              
0     144  19.28
1      51   6.83
2      37   4.95
3      68   9.10
4      65   8.70
5     382  51.14

=== valid flag ===
valid
True    747
Name: count, dtype: int64


In [15]:
# Utterances — Overlap issueId dengan Scored Sample
print('Utterances shape:', df_utterances.shape)
print('Kolom:', list(df_utterances.columns))
print()

# Kolom ID di utterances: 'issuedid' (per FEATURES.md)
utr_id_col    = 'issuedid' if 'issuedid' in df_utterances.columns else df_utterances.columns[0]
scored_id_col = 'id'       if 'id'       in df_scored.columns    else df_scored.columns[0]

utr_ids    = set(df_utterances[utr_id_col].dropna().astype(int).unique())
scored_ids = set(df_scored[scored_id_col].dropna().astype(int).unique())

overlap        = utr_ids & scored_ids
only_in_utr    = utr_ids - scored_ids
only_in_scored = scored_ids - utr_ids

print('Unique issueId di utterances   : {:,}'.format(len(utr_ids)))
print('Unique issueId di scored sample: {:,}'.format(len(scored_ids)))
print('Overlap (ada di keduanya)      : {:,}'.format(len(overlap)))
print('Hanya di utterances            : {:,}'.format(len(only_in_utr)))
print('Hanya di scored (tidak di utr) : {:,}'.format(len(only_in_scored)))
print()
if len(only_in_utr) == 0:
    print('Konklusi: SEMUA issueId di utterances ada di scored sample. Konsisten.')
else:
    print('Peringatan: {:,} issueId di utterances TIDAK ada di scored sample.'.format(len(only_in_utr)))
if len(only_in_scored) > 0:
    print('Info: {:,} tiket di scored sample tidak punya utterances (tidak masalah).'.format(len(only_in_scored)))

Utterances shape: (30104, 9)
Kolom: ['issueid', 'id', 'comment_seq', 'utr_seq', 'author', 'created', 'is_private', 'actionbody', 'author_role']

Unique issueId di utterances   : 360
Unique issueId di scored sample: 360
Overlap (ada di keduanya)      : 360
Hanya di utterances            : 0
Hanya di scored (tidak di utr) : 0

Konklusi: SEMUA issueId di utterances ada di scored sample. Konsisten.


### 2.2 Nilai Kosong - Pola dan Mekanisme

Kita menghitung berapa persen nilai yang kosong di setiap kolom untuk semua file, lalu mengidentifikasi apakah nilai kosong tersebut terjadi secara acak atau sistematis.

Penjelasan mekanisme:
- **MCAR** (Missing Completely At Random): nilai kosong terjadi secara acak, tidak berkaitan dengan kolom lain
- **MAR** (Missing At Random): nilai kosong berkaitan dengan nilai di kolom lain yang bisa diamati
- **MNAR** (Missing Not At Random): nilai kosong berkaitan langsung dengan nilai itu sendiri, misalnya tiket yang belum selesai tidak memiliki tanggal selesai

In [16]:
# Missing Values DS1
miss1 = df1.isnull().sum()
if miss1.sum() == 0:
    print('DS1: TIDAK ADA missing values (0 null di semua 10 kolom).')
    print('Dataset simulasi IBM Watson sudah lengkap — tidak perlu handling null.')
else:
    pct1 = (miss1 / len(df1) * 100).round(2)
    df_miss1 = pd.DataFrame({'missing_count': miss1, 'missing_pct': pct1})
    display(df_miss1.query('missing_count > 0').sort_values('missing_pct', ascending=False))

DS1: TIDAK ADA missing values (0 null di semua 10 kolom).
Dataset simulasi IBM Watson sudah lengkap — tidak perlu handling null.


In [17]:
# Missing Values DS2 issues.csv — tabel lengkap
miss2 = df_issues.isnull().sum()
pct2  = (miss2 / len(df_issues) * 100).round(2)
df_miss2 = (pd.DataFrame({'missing_count': miss2, 'missing_pct': pct2}).query('missing_count > 0').sort_values('missing_pct', ascending=False))
print('Kolom dengan missing (issues.csv): {}'.format(len(df_miss2)))
display(df_miss2)

Kolom dengan missing (issues.csv): 24


,missing_count,missing_pct
wf_in_review,66614,99.88
wf_rejected,66579,99.83
wf_deployment,66556,99.80
wf_cancelled,66531,99.76
wf_pending_customer_approval,66437,99.62
wf_testing_monitoring,66401,99.57
wf_monitoring,66173,99.22
wf_to_do,65906,98.82
wf_done,65831,98.71
wf_pending_deployment,65278,97.88


In [18]:
# Missing Values DS2 — change_history, snapshot, utterances

for label, df in [
    ('issues_change_history.csv', df_history),
    ('issues_snapshot.csv',       df_snapshot),
    ('sample_utterances.csv',     df_utterances),
]:
    miss = df.isnull().sum()
    pct  = (miss / len(df) * 100).round(2)
    df_m = (pd.DataFrame({'missing_count': miss, 'missing_pct': pct}).query('missing_count > 0').sort_values('missing_pct', ascending=False))
    print('\n=== ' + label + ' ===')
    if len(df_m) == 0:
        print('Tidak ada missing values.')
    else:
        print('Kolom dengan missing: {}'.format(len(df_m)))
        display(df_m.head(25))


=== issues_change_history.csv ===
Kolom dengan missing: 1


,missing_count,missing_pct
value,1840,0.71



=== issues_snapshot.csv ===
Kolom dengan missing: 24


,missing_count,missing_pct
wf_in_review,90877,99.91
wf_rejected,90850,99.88
wf_deployment,90814,99.84
wf_cancelled,90802,99.82
wf_pending_customer_approval,90692,99.70
wf_testing_monitoring,90658,99.66
wf_monitoring,90423,99.41
wf_to_do,90124,99.08
wf_done,90078,99.03
wf_pending_deployment,89439,98.32



=== sample_utterances.csv ===
Kolom dengan missing: 1


,missing_count,missing_pct
actionbody,23,0.08


In [19]:
# MAR Check — issue_resolution_date vs issue_status
# Hipotesis: null resolution_date berkorelasi dengan status tiket belum selesai

df_issues['_hasResDate'] = df_issues['issue_resolution_date'].notna()
ct_mar = pd.crosstab(
    df_issues['issue_status'],
    df_issues['_hasResDate'],
    normalize='index'
).round(4) * 100
ct_mar.columns = ['res_date_NULL_%', 'res_date_FILLED_%']
ct_mar_sorted  = ct_mar.sort_values('res_date_FILLED_%', ascending=False)
print('Cross-tab issue_status vs (resolution_date terisi):')
display(ct_mar_sorted)

df_issues.drop('_hasResDate', axis=1, inplace=True)
print()
print('Konklusi: jika res_date_FILLED_% tinggi hanya untuk status closed/done,')
print('maka null resolution_date adalah MAR (berkorelasi dengan status belum selesai).')

Cross-tab issue_status vs (resolution_date terisi):


,res_date_NULL_%,res_date_FILLED_%
issue_status,,
done,0.29,99.71
closed,0.34,99.66
approved,100.00,0.00
cancelled,100.00,0.00
in_progress,100.00,0.00
in_review,100.00,0.00
open,100.00,0.00
pending_deployment,100.00,0.00
reopened,100.00,0.00



Konklusi: jika res_date_FILLED_% tinggi hanya untuk status closed/done,
maka null resolution_date adalah MAR (berkorelasi dengan status belum selesai).


In [ ]:
# MCAR/MAR/MNAR Classification Table

class_data = [
    {
        'Dataset & Kolom':         'DS1 — semua kolom',
        'null_%':                  '0%',
        'Mekanisme':               'N/A',
        'Strategi Usulan':         'Tidak perlu handling',
        'Alasan':                  'DS1 simulasi lengkap, 0 null dikonfirmasi'
    },
    {
        'Dataset & Kolom':         'DS2 issue_assignee',
        'null_%':                  '~46.6%',
        'Mekanisme':               'MAR',
        'Strategi Usulan':         'Fill "unassigned"',
        'Alasan':                  'Tiket belum di-assign tidak punya assignee — berkorelasi dengan state awal'
    },
    {
        'Dataset & Kolom':         'DS2 issue_resolution_date',
        'null_%':                  '~1.3%',
        'Mekanisme':               'MAR',
        'Strategi Usulan':         'Flag isResolved + pertahankan null',
        'Alasan':                  'Null hanya pada tiket Open/In Progress — bukan data hilang (terbukti di cross-tab)'
    },
    {
        'Dataset & Kolom':         'DS2 issue_resolution',
        'null_%':                  '~1.3%',
        'Mekanisme':               'MAR',
        'Strategi Usulan':         'Fill "Unknown"',
        'Alasan':                  'Satu paket dengan issue_resolution_date'
    },
    {
        'Dataset & Kolom':         'DS2 wf_* (16 kolom, >95% null)',
        'null_%':                  '95-100%',
        'Mekanisme':               'MNAR',
        'Strategi Usulan':         'Drop; retain wf_resolved/in_progress/waiting/open + semua wfe_*',
        'Alasan':                  'Null = tiket tidak pernah lewat state itu — MNAR by design, bukan data hilang'
    },
    {
        'Dataset & Kolom':         'DS2 wf_resolved (~57.8%)',
        'null_%':                  '~57.8%',
        'Mekanisme':               'MNAR',
        'Strategi Usulan':         'Fill 0',
        'Alasan':                  'Null = 0 detik di state resolved (tidak pernah resolved)'
    },
    {
        'Dataset & Kolom':         'DS2 wf_in_progress (~44.9%)',
        'null_%':                  '~44.9%',
        'Mekanisme':               'MNAR',
        'Strategi Usulan':         'Fill 0',
        'Alasan':                  'Sama — null = tidak pernah in_progress'
    },
    {
        'Dataset & Kolom':         'DS2 wf_waiting (~70.5%)',
        'null_%':                  '~70.5%',
        'Mekanisme':               'MNAR',
        'Strategi Usulan':         'Fill 0',
        'Alasan':                  'Sama'
    },
    {
        'Dataset & Kolom':         'DS2 wf_open (~0.1%)',
        'null_%':                  '~0.1%',
        'Mekanisme':               'MCAR',
        'Strategi Usulan':         'Fill 0',
        'Alasan':                  'Sangat sedikit, tidak berpola — acak'
    },
    {
        'Dataset & Kolom':         'DS2 snapshot — kolom wf_* (sama dengan issues)',
        'null_%':                  'Sama pola',
        'Mekanisme':               'MNAR',
        'Strategi Usulan':         'Sama dengan issues.csv',
        'Alasan':                  'Snapshot adalah subset dari issues dengan granularitas per-turn'
    },
]
df_class = pd.DataFrame(class_data)
print('=== Tabel Klasifikasi Missing Value ===')
display(df_class)
print()
print('Legenda: MCAR=Missing Completely At Random | MAR=Missing At Random | MNAR=Missing Not At Random')

=== Tabel Klasifikasi Missing Value ===


,Dataset & Kolom,null_%,Mekanisme,Strategi Usulan,Alasan
0,DS1 — semua kolom,0%,N/A,Tidak perlu handling,"DS1 simulasi lengkap, 0 null dikonfirmasi"
1,DS2 issue_assignee,~46.6%,MAR,"Fill ""unassigned""",Tiket belum di-assign tidak punya assignee — b...
2,DS2 issue_resolution_date,~1.3%,MAR,Flag isResolved + pertahankan null,Null hanya pada tiket Open/In Progress — bukan...
3,DS2 issue_resolution,~1.3%,MAR,"Fill ""Unknown""",Satu paket dengan issue_resolution_date
4,"DS2 wf_* (16 kolom, >95% null)",95-100%,MNAR,Drop; retain wf_resolved/in_progress/waiting/o...,Null = tiket tidak pernah lewat state itu — MN...
5,DS2 wf_resolved (~57.8%),~57.8%,MNAR,Fill 0,Null = 0 detik di state resolved (tidak pernah...
6,DS2 wf_in_progress (~44.9%),~44.9%,MNAR,Fill 0,Sama — null = tidak pernah in_progress
7,DS2 wf_waiting (~70.5%),~70.5%,MNAR,Fill 0,Sama
8,DS2 wf_open (~0.1%),~0.1%,MCAR,Fill 0,"Sangat sedikit, tidak berpola — acak"
9,DS2 snapshot — kolom wf_* (sama dengan issues),Sama pola,MNAR,Sama dengan issues.csv,Snapshot adalah subset dari issues dengan gran...



Legenda: MCAR=Missing Completely At Random | MAR=Missing At Random | MNAR=Missing Not At Random


### 2.3 Pemeriksaan Duplikasi

Kita memeriksa apakah ada baris yang identik persis (duplikat penuh) atau tiket yang terdaftar lebih dari satu kali. Baris duplikat perlu dihapus agar tidak menggandakan pengaruh tiket tertentu pada hasil analisis.

In [21]:
# DS1 — Duplikasi
dup_all    = df1.duplicated().sum()
dup_ticket = df1.duplicated(subset=['ticket']).sum()
print('DS1 — baris 100% identik    : {} ({:.2f}%)'.format(dup_all, dup_all/len(df1)*100))
print('DS1 — duplikasi by ticket   : {}'.format(dup_ticket))
print('DS1 — total baris           : {:,}'.format(len(df1)))
if dup_all == 0:
    print('\nKonklusi: Tidak ada duplikasi di DS1.')
else:
    print('\nPeringatan: ada {:,} baris duplikat — akan di-drop di Scrub Awal.'.format(dup_all))

DS1 — baris 100% identik    : 0 (0.00%)
DS1 — duplikasi by ticket   : 0
DS1 — total baris           : 100,000

Konklusi: Tidak ada duplikasi di DS1.


In [22]:
# DS2 issues — Unique Ticket ID Check
n_rows_iss   = len(df_issues)
n_unique_iss = df_issues['id'].nunique()
dup_full_iss = df_issues.duplicated().sum()
dup_id_iss   = df_issues.duplicated(subset=['id']).sum()

print('DS2 issues — total baris     : {:,}'.format(n_rows_iss))
print('DS2 issues — unique id       : {:,}'.format(n_unique_iss))
print('DS2 issues — duplikasi penuh : {}'.format(dup_full_iss))
print('DS2 issues — duplikasi by id : {}'.format(dup_id_iss))

if n_rows_iss == n_unique_iss and dup_id_iss == 0:
    print('\nKonklusi: id adalah primary key unik di issues.csv — tidak ada duplikasi.')
else:
    print('\nPeringatan: {:,} duplikasi berdasarkan id — perlu investigasi.'.format(dup_id_iss))

DS2 issues — total baris     : 66,691
DS2 issues — unique id       : 66,691
DS2 issues — duplikasi penuh : 0
DS2 issues — duplikasi by id : 0

Konklusi: id adalah primary key unik di issues.csv — tidak ada duplikasi.


In [23]:
# DS2 Snapshot — Verifikasi Desain Per-Assignee
print('DS2 snapshot shape : {:,} x {}'.format(df_snapshot.shape[0], df_snapshot.shape[1]))
print('DS2 issues shape   : {:,} x {}'.format(df_issues.shape[0], df_issues.shape[1]))

ratio = df_snapshot.shape[0] / df_issues.shape[0]
print('\nRasio baris snapshot / issues: {:.2f}x'.format(ratio))
print('Ekspektasi: >1.0 karena 1 issue bisa punya lebih dari 1 assignee (turn)')

# Hitung distribusi jumlah turn per Issue
snap_id_col = 'id' if 'id' in df_snapshot.columns else df_snapshot.columns[1]
turns_per_issue = df_snapshot.groupby(snap_id_col).size()
multi_turn = (turns_per_issue > 1).sum()

print('\nIssue dengan >1 turn (multi-assignee) : {:,}'.format(multi_turn))
print('Max turn per Issue                    : {}'.format(turns_per_issue.max()))
print('\nDistribusi jumlah turn per Issue:')
print(turns_per_issue.value_counts().sort_index().head(10))

# DS2 change_history: cek struktur
print('\n=== issues_change_history.csv — quick check ===')
print('Shape: {:,} x {}'.format(df_history.shape[0], df_history.shape[1]))
print('Kolom:', list(df_history.columns))
if 'field' in df_history.columns:
    print('\nValue counts field (jenis perubahan):')
    print(df_history['field'].value_counts())

print('\nKonklusi: Duplikasi di Snapshot adalah DESAIN bukan Error.')
print('Setiap baris = 1 turn assignee pada 1 issue')

DS2 snapshot shape : 90,963 x 60
DS2 issues shape   : 66,691 x 58

Rasio baris snapshot / issues: 1.36x
Ekspektasi: >1.0 karena 1 issue bisa punya lebih dari 1 assignee (turn)



Issue dengan >1 turn (multi-assignee) : 16,536
Max turn per Issue                    : 15

Distribusi jumlah turn per Issue:
1     50155
2     11147
3      3804
4      1094
5       341
6        84
7        39
8        17
9         3
10        3
Name: count, dtype: int64

=== issues_change_history.csv — quick check ===
Shape: 257,508 x 6
Kolom: ['id', 'issueid', 'field', 'value', 'created', 'change_group_id']

Value counts field (jenis perubahan):
field
status      194802
assignee     62706
Name: count, dtype: int64

Konklusi: Duplikasi di Snapshot adalah DESAIN bukan Error.
Setiap baris = 1 turn assignee pada 1 issue


### 2.4 Tipe Data dan Format

Kita memeriksa tipe data setiap kolom dan memastikan formatnya sudah sesuai. Kolom tanggal harus bertipe datetime, kolom angka harus bertipe numerik, dan nilai teks harus konsisten dalam hal huruf besar/kecil dan spasi.

In [ ]:
# DS1 — Tipe Data & Format
print('=== DS1 Dtypes ===')
print(df1.dtypes)
print()
print('=== Format Kolom Kategorikal DS1 (sample values) ===')
for col in ['RequestorSeniority', 'Severity', 'Priority', 'Satisfaction']:
    vals = df1[col].unique()[:5]
    print('  {}: {}'.format(col, vals))
print()
print('Temuan: semua kolom kategorikal DS1 berformat "N - Label".')
print('Tidak ada kolom timestamp di DS1 — hanya daysOpen sebagai proxy durasi hari.')
print('Strategi Scrub: parse integer N dan string Label terpisah (feature engineering).')

=== DS1 Dtypes ===
ticket                int64
requestor             int64
RequestorSeniority      str
ITOwner               int64
FiledAgainst            str
TicketType              str
Severity                str
Priority                str
daysOpen              int64
Satisfaction            str
dtype: object

=== Format Kolom Kategorikal DS1 (sample values) ===


  RequestorSeniority: <StringArray>
['1 - Junior', '2 - Regular', '4 - Management', '3 - Senior']
Length: 4, dtype: str
  Severity: <StringArray>
['2 - Normal', '1 - Minor', '3 - Major', '4 - Critical', '0 - Unclassified']
Length: 5, dtype: str
  Priority: <StringArray>
['0 - Unassigned', '1 - Low', '3 - High', '2 - Medium']
Length: 4, dtype: str
  Satisfaction: <StringArray>
['1 - Unsatisfied', '0 - Unknown', '3 - Highly satisfied', '2 - Satisfied']
Length: 4, dtype: str

Temuan: semua kolom kategorikal DS1 berformat "N - Label".
Tidak ada kolom timestamp di DS1 — hanya daysOpen sebagai proxy durasi hari.
Strategi Scrub: parse integer N dan string Label terpisah (feature engineering).


In [25]:
# DS2 — Kolom Timestamp & Format Check

TS_KEYWORDS = ['date', 'created', 'started', 'ended', 'last', 'change']

for label, df in [
    ('issues.csv',               df_issues),
    ('issues_change_history.csv', df_history),
    ('issues_snapshot.csv',       df_snapshot),
    ('sample_utterances.csv',     df_utterances),
]:
    ts_cols = [c for c in df.columns
               if any(kw in c.lower() for kw in TS_KEYWORDS)]
    print('\n=== ' + label + ' — timestamp columns ===')
    if not ts_cols:
        print('  Tidak ada kolom timestamp.')
    for col in ts_cols:
        dtype  = str(df[col].dtype)
        sample = str(df[col].dropna().iloc[0]) if df[col].notna().any() else 'ALL NULL'
        print('  {:<35}: dtype={:<10}  sample="{}"'.format(col, dtype, sample[:40]))

print('\nTemuan: semua kolom timestamp di DS2 masih object dtype.')
print('Strategi Scrub: konversi dengan pd.to_datetime(format="ISO8601", utc=True).')


=== issues.csv — timestamp columns ===
  started                            : dtype=str         sample="2016-01-06 08:23:43+00:00"
  ended                              : dtype=str         sample="2016-01-06 08:56:55+00:00"
  issue_created                      : dtype=str         sample="2016-01-06 08:23:43+00:00"


  issue_resolution_date              : dtype=str         sample="2016-01-06 08:56:55+00:00"
  last_change_date                   : dtype=str         sample="2016-04-02 12:20:21+00:00"

=== issues_change_history.csv — timestamp columns ===
  created                            : dtype=str         sample="2016-03-24 15:35:53+00:00"
  change_group_id                    : dtype=float64     sample="10707.0"

=== issues_snapshot.csv — timestamp columns ===
  started                            : dtype=str         sample="2016-01-06 08:23:43+00:00"
  ended                              : dtype=str         sample="2016-01-06 08:56:55+00:00"
  issue_created                      : dtype=str         sample="2016-01-06 08:23:43+00:00"


  issue_resolution_date              : dtype=str         sample="2016-01-06 08:56:55+00:00"
  last_change_date                   : dtype=str         sample="2016-04-02 12:20:21+00:00"

=== sample_utterances.csv — timestamp columns ===
  created                            : dtype=str         sample="2022-01-02 11:12:51+00:00"

Temuan: semua kolom timestamp di DS2 masih object dtype.
Strategi Scrub: konversi dengan pd.to_datetime(format="ISO8601", utc=True).


In [26]:
# Satuan wf_total_time & Estimasi Time Per Step

print('=== wf_total_time (DETIK, per FEATURES.md) ===')
stats_t = df_issues['wf_total_time'].describe()
print(stats_t.round(0))
print()
print('Konversi ke JAM:')
print('  Median : {:.1f} jam'.format(stats_t['50%'] / 3600))
print('  P75    : {:.1f} jam'.format(df_issues['wf_total_time'].quantile(0.75) / 3600))
print('  Max    : {:.1f} jam ({:.1f} hari)'.format(stats_t['max'] / 3600, stats_t['max'] / 86400))

# Estimasi timePerStep
mask_valid = df_issues['processing_steps'] > 0
n_valid    = mask_valid.sum()
timePerStepEst = (df_issues.loc[mask_valid, 'wf_total_time']
                  / df_issues.loc[mask_valid, 'processing_steps'])

print()
print('=== Estimasi Time Per Step (detik, dari {:,} tiket) ==='.format(n_valid))
print(timePerStepEst.describe().round(0))
print('Median per step: {:.1f} jam'.format(timePerStepEst.median() / 3600))

print()
print('Satuan dikonfirmasi: DETIK.')
print('Strategi Scrub: buat kolom totalTimeHours = wf_total_time / 3600.')

=== wf_total_time (DETIK, per FEATURES.md) ===


count       66691.00
mean     65856465.00
std      99715344.00
min             0.00
25%        440845.00
50%       2961971.00
75%     129318977.00
max     357193930.00
Name: wf_total_time, dtype: float64

Konversi ke JAM:
  Median : 822.8 jam
  P75    : 35921.9 jam
  Max    : 99220.5 jam (4134.2 hari)

=== Estimasi Time Per Step (detik, dari 66,613 tiket) ===
count       66613.00
mean     62956377.00
std      99673664.00
min             3.00
25%        115782.00
50%        727859.00
75%     122957828.00
max     357193930.00
dtype: float64
Median per step: 202.2 jam

Satuan dikonfirmasi: DETIK.
Strategi Scrub: buat kolom totalTimeHours = wf_total_time / 3600.


### 2.5 Deteksi Outlier

Metode yang digunakan adalah IQR (Interquartile Range). Nilai dianggap outlier jika berada di bawah Q1 - 1.5 x IQR atau di atas Q3 + 1.5 x IQR.

Penting untuk dibedakan antara outlier karena kesalahan data (perlu dihapus atau diperbaiki) dan outlier karena memang tiket yang sangat kompleks atau butuh waktu sangat lama (perlu dibatasi nilainya tapi tidak dihapus dari dataset).

In [27]:
# IQR Outlier Detection — Week 03 Method
def iqrOutlier(series, col_name, verbose=True):
    """Hitung outlier dengan IQR method."""
    clean  = series.dropna()
    Q1     = clean.quantile(0.25)
    Q3     = clean.quantile(0.75)
    IQR    = Q3 - Q1
    lower  = Q1 - 1.5 * IQR
    upper  = Q3 + 1.5 * IQR
    out    = clean[(clean < lower) | (clean > upper)]
    if verbose:
        print('[{}]'.format(col_name))
        print('  Q1={:.2f}, Q3={:.2f}, IQR={:.2f}'.format(Q1, Q3, IQR))
        print('  Fence: [{:.2f}, {:.2f}]'.format(lower, upper))
        print('  Outlier: {:,} dari {:,} ({:.2f}%)'.format(len(out), len(clean), len(out)/len(clean)*100))
        if len(out) > 0:
            print('  Range outlier: [{:.2f}, {:.2f}]'.format(out.min(), out.max()))
            neg = (out < 0).sum()
            if neg > 0:
                print('  PERINGATAN: {} nilai negatif -> kemungkinan data error!'.format(neg))
    return lower, upper, out

print('Fungsi iqrOutlier siap digunakan.')

Fungsi iqrOutlier siap digunakan.


In [28]:
# DS1 — daysOpen Outlier
print('=== DS1: daysOpen ===')
lo1, hi1, out1 = iqrOutlier(df1['daysOpen'], 'daysOpen')
p95_days = df1['daysOpen'].quantile(0.95)
p99_days = df1['daysOpen'].quantile(0.99)
print('  P95: {:.0f} hari | P99: {:.0f} hari'.format(p95_days, p99_days))
print()
# [AI - assisted]
print('Konklusi: outlier daysOpen adalah genuine extreme (tiket lama terbuka), bukan error.')
print('Strategi Scrub: cap P99; buat flag isLongTicket jika > P95.')
print()
print('Catatan: DS1 tidak punya reassignmentCount/reopenCount/sysModCount.')
print('(Kolom-kolom tersebut hanya ada di UCI Incident Log, bukan dataset ini.)')

=== DS1: daysOpen ===
[daysOpen]
  Q1=1.00, Q3=10.00, IQR=9.00
  Fence: [-12.50, 23.50]
  Outlier: 4,014 dari 100,000 (4.01%)
  Range outlier: [24.00, 54.00]
  P95: 22 hari | P99: 31 hari

Konklusi: outlier daysOpen adalah genuine extreme (tiket lama terbuka), bukan error.
Strategi Scrub: cap P99; buat flag isLongTicket jika > P95.

Catatan: DS1 tidak punya reassignmentCount/reopenCount/sysModCount.
(Kolom-kolom tersebut hanya ada di UCI Incident Log, bukan dataset ini.)


In [29]:
# DS2 — Outlier Kolom Numerik Utama
print('=== wf_total_time (detik) ===')
lo_t, hi_t, out_t = iqrOutlier(df_issues['wf_total_time'], 'wf_total_time')
p99_t = df_issues['wf_total_time'].quantile(0.99)
print('  P99: {:.1f} jam'.format(p99_t / 3600))

print()
print('=== processing_steps ===')
lo_p, hi_p, out_p = iqrOutlier(df_issues['processing_steps'], 'processing_steps')
print('  P99: {:.0f} steps'.format(df_issues['processing_steps'].quantile(0.99)))

print()
print('=== issue_comments_count ===')
lo_c, hi_c, out_c = iqrOutlier(df_issues['issue_comments_count'], 'issue_comments_count')

print()
print('=== wfe_reopened (jumlah re-open) ===')
lo_r, hi_r, out_r = iqrOutlier(df_issues['wfe_reopened'], 'wfe_reopened')

print()
print('=== issue_contr_count (jumlah kontributor) ===')
lo_k, hi_k, out_k = iqrOutlier(df_issues['issue_contr_count'], 'issue_contr_count')

print()
print('Konklusi')
print('- wf_total_time dan processing_steps: outlier genuine extreme (tiket kompleks)')
print('- wfe_reopened tinggi: tiket yang dibuka ulang berkali-kali -> indikator kompleksitas')
print('- Tidak ada nilai negatif yang terindikasi error (akan dikonfirmasi setelah parse timestamp)')
print('- Strategi Scrub: cap P99 untuk wf_total_time; buat flag isComplex dari wfe_reopened')

=== wf_total_time (detik) ===
[wf_total_time]
  Q1=440845.37, Q3=129318977.03, IQR=128878131.67
  Fence: [-192876352.14, 322636174.54]
  Outlier: 895 dari 66,691 (1.34%)
  Range outlier: [322695386.25, 357193929.88]
  P99: 90910.6 jam

=== processing_steps ===
[processing_steps]
  Q1=1.00, Q3=4.00, IQR=3.00
  Fence: [-3.50, 8.50]
  Outlier: 2,802 dari 66,691 (4.20%)
  Range outlier: [9.00, 93.00]
  P99: 13 steps

=== issue_comments_count ===
[issue_comments_count]
  Q1=3.00, Q3=10.00, IQR=7.00
  Fence: [-7.50, 20.50]
  Outlier: 5,314 dari 66,691 (7.97%)
  Range outlier: [21.00, 685.00]

=== wfe_reopened (jumlah re-open) ===
[wfe_reopened]
  Q1=0.00, Q3=0.00, IQR=0.00
  Fence: [0.00, 0.00]
  Outlier: 2,123 dari 66,691 (3.18%)
  Range outlier: [1.00, 11.00]

=== issue_contr_count (jumlah kontributor) ===
[issue_contr_count]
  Q1=1.00, Q3=1.00, IQR=0.00
  Fence: [1.00, 1.00]
  Outlier: 11,647 dari 66,691 (17.46%)
  Range outlier: [2.00, 10.00]

Konklusi
- wf_total_time dan processing_step

In [30]:
# DS2 — Durasi Resolusi Sementara (cek outlier sebelum cleaning formal)
df_temp = df_issues[['issue_created', 'issue_resolution_date']].copy()
df_temp['_createdDt']  = pd.to_datetime(df_temp['issue_created'],
                                        format='ISO8601', utc=True, errors='coerce')
df_temp['_resolvedDt'] = pd.to_datetime(df_temp['issue_resolution_date'],
                                        format='ISO8601', utc=True, errors='coerce')
df_temp['_resDurHours'] = (
    (df_temp['_resolvedDt'] - df_temp['_createdDt']).dt.total_seconds() / 3600
)

resolved_only = df_temp['_resDurHours'].dropna()
print('Tiket dengan durasi resolusi terhitung: {:,}'.format(len(resolved_only)))
print()
print('=== Durasi Resolusi Sementara (jam) ===')
print(resolved_only.describe().round(2))

neg_count = (resolved_only < 0).sum()
print()
if neg_count > 0:
    print('PERINGATAN: {:,} nilai negatif (resolved sebelum created) -> data error!'.format(neg_count))
else:
    print('Nilai negatif: 0 -> no Data Error Durasi.')

print()
print('=== IQR Durasi Resolusi ===')
lo_d, hi_d, out_d = iqrOutlier(resolved_only, 'resolutionDurationHours')
p99_d = resolved_only.quantile(0.99)
print('  P99: {:.1f} jam ({:.1f} hari)'.format(p99_d, p99_d / 24))
print()
print('Strategi Scrub: cap P99 di Scrub Lanjutan.')
del df_temp

Tiket dengan durasi resolusi terhitung: 65,838

=== Durasi Resolusi Sementara (jam) ===
count   65838.00
mean    18482.55
std     27805.94
min         0.00
25%       124.32
50%       837.81
75%     36508.15
max     99220.54
Name: _resDurHours, dtype: float64

Nilai negatif: 0 -> no Data Error Durasi.

=== IQR Durasi Resolusi ===
[resolutionDurationHours]
  Q1=124.32, Q3=36508.15, IQR=36383.83
  Fence: [-54451.42, 91083.89]
  Outlier: 636 dari 65,838 (0.97%)
  Range outlier: [91084.92, 99220.54]
  P99: 90929.0 jam (3788.7 hari)

Strategi Scrub: cap P99 di Scrub Lanjutan.


### 2.6 Ketidakseimbangan Kelas (Class Imbalance)

Kita memeriksa apakah distribusi nilai pada kolom yang akan dijadikan target klasifikasi sudah seimbang atau tidak. Ketidakseimbangan yang ekstrem (rasio lebih dari 10:1) bisa menyebabkan model cenderung memprediksi kelas mayoritas saja dan mengabaikan kelas minoritas.

Aturan yang digunakan:
- SMOTE (teknik oversampling sintetis) hanya diterapkan jika kolom digunakan sebagai target klasifikasi DAN rasio kelas lebih besar dari 10:1
- SMOTE hanya dieksekusi pada data training, bukan seluruh dataset, untuk mencegah kebocoran data (data leakage)
- Untuk analisis eksplorasi atau clustering, tidak perlu melakukan penyeimbangan kelas

In [31]:
# DS1 — Class Imbalance Severity, Priority, Satisfaction

for col in ['Severity', 'Priority', 'Satisfaction']:
    vc    = df1[col].value_counts()
    pct   = (vc / len(df1) * 100).round(2)
    tbl   = pd.DataFrame({'count': vc, 'pct_%': pct})
    ratio = vc.max() / vc.min() if vc.min() > 0 else float('inf')
    print('=== DS1: {} ==='.format(col))
    print(tbl.to_string())
    # [AI-assisted]
    print('Rasio mayoritas/minoritas: {:.1f}:1'.format(ratio))
    if ratio > 10:
        print('-> IMBALANCED SIGNIFIKAN (>10:1). Pertimbangkan class_weight atau SMOTE di train split.')
    elif ratio > 3:
        print('-> Moderately imbalanced. Gunakan stratified split + class_weight.')
    else:
        print('-> Distribusi cukup merata. Stratified split cukup.')
    print()

=== DS1: Severity ===
                  count  pct_%
Severity                      
2 - Normal        90912  90.91
3 - Major          4974   4.97
1 - Minor          2317   2.32
4 - Critical       1430   1.43
0 - Unclassified    367   0.37
Rasio mayoritas/minoritas: 247.7:1
-> IMBALANCED SIGNIFIKAN (>10:1). Pertimbangkan class_weight atau SMOTE di train split.

=== DS1: Priority ===
                count  pct_%
Priority                    
3 - High        36498  36.50
0 - Unassigned  30127  30.13
1 - Low         17117  17.12
2 - Medium      16258  16.26
Rasio mayoritas/minoritas: 2.2:1
-> Distribusi cukup merata. Stratified split cukup.

=== DS1: Satisfaction ===
                      count  pct_%
Satisfaction                      
0 - Unknown           30211  30.21
3 - Highly satisfied  29063  29.06
1 - Unsatisfied       21124  21.12
2 - Satisfied         19602  19.60
Rasio mayoritas/minoritas: 1.5:1
-> Distribusi cukup merata. Stratified split cukup.



In [32]:
# DS2 issues — issue_priority Imbalance
vc_p  = df_issues['issue_priority'].value_counts(dropna=False)
pct_p = (vc_p / len(df_issues) * 100).round(2)
tbl_p = pd.DataFrame({'count': vc_p, 'pct_%': pct_p})
print('=== DS2: issue_priority (n={:,}) ==='.format(len(df_issues)))
print(tbl_p.to_string())

# Analisis tanpa "unknown"
no_unk = df_issues[df_issues['issue_priority'] != 'unknown']['issue_priority']
if len(no_unk) > 0:
    vc_nu  = no_unk.value_counts()
    rat_nu = vc_nu.max() / vc_nu.min() if vc_nu.min() > 0 else float('inf')
    print()
    print('=== Tanpa "unknown" ({:,} tiket) ==='.format(len(no_unk)))
    print(pd.DataFrame({'count': vc_nu, 'pct_%': (vc_nu / len(no_unk) * 100).round(2)}).to_string())
    print('Rasio majoritas/minoritas (tanpa unknown): {:.1f}:1'.format(rat_nu))

print()
print('Konklusi: "unknown" sangat dominan -> perlu normalisasi priority di Scrub.')
print('Setelah normalisasi, cek ulang rasio untuk keputusan SMOTE.')

=== DS2: issue_priority (n=66,691) ===
                count  pct_%
issue_priority              
unknown         33965  50.93
Medium          24788  37.17
High             4554   6.83
Highest          2084   3.12
Blocker           656   0.98
Low               560   0.84
Lowest             84   0.13



=== Tanpa "unknown" (32,726 tiket) ===
                count  pct_%
issue_priority              
Medium          24788  75.74
High             4554  13.92
Highest          2084   6.37
Blocker           656   2.00
Low               560   1.71
Lowest             84   0.26
Rasio majoritas/minoritas (tanpa unknown): 295.1:1

Konklusi: "unknown" sangat dominan -> perlu normalisasi priority di Scrub.
Setelah normalisasi, cek ulang rasio untuk keputusan SMOTE.


In [33]:
# Scored Sample — Q1/Q2/Q3 Imbalance & Binary Threshold

for q in ['Q1', 'Q2', 'Q3']:
    if q not in df_scored.columns:
        continue
    q_clean = df_scored[q].dropna()
    vc      = q_clean.value_counts().sort_index()
    pct     = (vc / vc.sum() * 100).round(2)
    tbl     = pd.DataFrame({'count': vc, 'pct_%': pct})

    good    = (q_clean >= 4).sum()
    bad     = (q_clean < 4).sum()
    total   = good + bad
    ratio   = good / bad if bad > 0 else float('inf')

    print('=== {} (n={}) ==='.format(q, total))
    print(tbl.to_string())
    print()
    print('  Binary >=4 (good) / <4 (needs_improvement):')
    print('    good: {} ({:.1f}%) | needs_improvement: {} ({:.1f}%)'.format(
        good, good/total*100, bad, bad/total*100))
    print('    Rasio good/bad: {:.2f}:1'.format(ratio))
    if ratio > 10:
        rec = 'SMOTE pada training split direkomendasikan (ratio >10:1)'
    elif ratio > 3:
        rec = 'Class_weight cukup (ratio 3-10:1)'
    else:
        rec = 'Distribusi cukup merata'
    print('    Rekomendasi: {}'.format(rec))
    print()

print('CATATAN: SMOTE HANYA dieksekusi pada training split di tahap Model, BUKAN di sini.')

=== Q1 (n=747) ===
    count  pct_%
Q1              
0     144  19.28
1      50   6.69
2      42   5.62
3      67   8.97
4      72   9.64
5     372  49.80

  Binary >=4 (good) / <4 (needs_improvement):
    good: 444 (59.4%) | needs_improvement: 303 (40.6%)
    Rasio good/bad: 1.47:1
    Rekomendasi: Distribusi cukup merata

=== Q2 (n=747) ===
    count  pct_%
Q2              
0     144  19.28
1      51   6.83
2      42   5.62
3      72   9.64
4      70   9.37
5     368  49.26

  Binary >=4 (good) / <4 (needs_improvement):
    good: 438 (58.6%) | needs_improvement: 309 (41.4%)
    Rasio good/bad: 1.42:1
    Rekomendasi: Distribusi cukup merata

=== Q3 (n=747) ===
    count  pct_%
Q3              
0     144  19.28
1      51   6.83
2      37   4.95
3      68   9.10
4      65   8.70
5     382  51.14

  Binary >=4 (good) / <4 (needs_improvement):
    good: 447 (59.8%) | needs_improvement: 300 (40.2%)
    Rasio good/bad: 1.49:1
    Rekomendasi: Distribusi cukup merata

CATATAN: SMOTE HANYA d

## 3. Ringkasan Temuan dan Rencana Pembersihan

Berdasarkan semua pemeriksaan diagnostik di atas, bagian ini merangkum semua masalah yang ditemukan dan strategi pembersihan yang akan diterapkan. Setiap langkah pembersihan di bagian berikutnya mengacu pada temuan yang tercantum di sini.

In [ ]:
# CHECKPOINT Ringkasan Temuan & Keputusan Scrub

findings = [
    ('1A', 'DS1: Severity dominasi "2-Normal" (~90%)',
     'class_weight jika jadi target; cek P99 setelah parse'),
    ('1A', 'DS1: Priority distribusi lebih merata',
     'stratified split cukup'),
    ('1A', 'DS1: Priority TIDAK auto-derived dari Severity',
     'cross-tab disimpan; tidak ada inkonsistensi'),
    ('1A', 'DS2: issue_priority "unknown" ~51%',
     'normalisasi ke high/medium/low/unknown di Scrub Awal'),
    ('1A', 'DS2: wf_total_time satuan DETIK',
     'bagi 3600 -> totalTimeHours'),
    ('1A', 'Scored: Q1/Q2/Q3 skala 1-5 tersedia',
     'binary split >=4=good; cek rasio lagi setelah cleaning'),
    ('1A', 'Utterances <-> scored: overlap dikonfirmasi',
     'konsisten sesuai desain FEATURES.md'),
    ('1B', 'DS1: 0% null di semua kolom',
     'tidak perlu handling'),
    ('1B', 'DS2 issue_assignee 46.6% null',
     'Fill "unassigned" (MAR by workflow)'),
    ('1B', 'DS2 issue_resolution_date 1.3% null',
     'Flag isResolved + pertahankan null (MAR)'),
    ('1B', 'DS2 wf_* >95% null (16 kolom)',
     'Drop; retain wf_resolved/in_progress/waiting/open + wfe_*'),
    ('1B', 'DS2 wf_resolved/in_progress/waiting: MNAR',
     'Fill 0 (null = tidak pernah lewat state itu)'),
    ('1C', 'DS1: tidak ada duplikasi penuh',
     'tidak perlu drop'),
    ('1C', 'DS2 issues: id adalah PK unik',
     'tidak ada duplikasi'),
    ('1C', 'DS2 snapshot: multi-row adalah desain per-turn',
     'dokumentasikan; JANGAN deduplicate'),
    ('1D', 'DS1: tidak ada timestamp; hanya daysOpen',
     'tidak bisa buat tren waktu untuk DS1'),
    ('1D', 'DS2: semua timestamp masih object dtype',
     'konversi pd.to_datetime() ISO8601 utc=True di Scrub Awal'),
    ('1D', 'DS1: format kolom kategorikal "N - Label"',
     'parse integer + string label terpisah di Feature Eng'),
    ('1E', 'DS1 daysOpen: outlier genuine extreme',
     'cap P99; flag isLongTicket > P95'),
    ('1E', 'DS2 wf_total_time, processing_steps: outlier extreme',
     'cap P99 di Scrub Lanjutan'),
    ('1E', 'DS2 durasi resolusi: tidak ada nilai negatif',
     'tidak perlu NaN-fy; cap P99 cukup'),
    ('1F', 'DS1 Severity: imbalanced (cek ratio setelah parse)',
     'class_weight jika >10:1; SMOTE hanya di train split'),
    ('1F', 'DS2 priority: "unknown" dominan sebelum normalisasi',
     'normalisasi dulu; cek ulang rasio'),
    ('1F', 'Scored Q: binary split tersedia',
     'SMOTE atau class_weight di Model — BUKAN di Scrub'),
]

print('=' * 96)
print('  CHECKPOINT RINGKASAN EDA DIAGNOSTIK')
print('=' * 96)
print('{:<5} {:<52} {:<38}'.format('FASE', 'TEMUAN', 'KEPUTUSAN SCRUB'))
print('-' * 96)
for fase, temuan, keputusan in findings:
    print('{:<5} {:<52} {:<38}'.format(fase, temuan[:51], keputusan[:37]))

print()
print('=' * 96)
print('Lanjut -> Scrub Awal (basic cleaning berdasarkan temuan di atas)')
print('=' * 96)

  CHECKPOINT RINGKASAN EDA DIAGNOSTIK
FASE  TEMUAN                                               KEPUTUSAN SCRUB                       
------------------------------------------------------------------------------------------------
1A    DS1: Severity dominasi "2-Normal" (~90%)             class_weight jika jadi target; cek P9 
1A    DS1: Priority distribusi lebih merata                stratified split cukup                
1A    DS1: Priority TIDAK auto-derived dari Severity       cross-tab disimpan; tidak ada inkonsi 
1A    DS2: issue_priority "unknown" ~51%                   normalisasi ke high/medium/low/unknow 
1A    DS2: wf_total_time satuan DETIK                      bagi 3600 -> totalTimeHours           
1A    Scored: Q1/Q2/Q3 skala 1-5 tersedia                  binary split >=4=good; cek rasio lagi 
1A    Utterances <-> scored: overlap dikonfirmasi          konsisten sesuai desain FEATURES.md   
1B    DS1: 0% null di semua kolom                          tidak perlu handling  

## 4. Tahap Pembersihan 1: Pembersihan Dasar

Tahap ini menangani masalah yang paling mendasar: menghapus duplikat dan menyeragamkan format teks. Setiap langkah mencatat berapa baris yang terpengaruh sebagai bukti bahwa pembersihan berhasil dilakukan.

### 4.1 DS1 - Pembersihan Awal

**Catatan penting tentang DS1:** Dataset utama yang digunakan adalah IBM Watson IT Help Desk (`WA_Fn-UseC_-IT-Help-Desk.csv`), bukan dataset UCI ServiceNow. Kolom-kolom yang tersedia adalah `ticket`, `requestor`, `RequestorSeniority`, `ITOwner`, `FiledAgainst`, `TicketType`, `Severity`, `Priority`, `daysOpen`, dan `Satisfaction`. Kolom seperti `incidentState`, `category`, `urgency`, dan `openedAt` tidak ada di dataset ini.

In [35]:
# Drop Baris 100% Identik — DS1
n_before_ds1 = len(df1)
df1 = df1.drop_duplicates().reset_index(drop=True)
n_dropped_ds1 = n_before_ds1 - len(df1)
print('DS1 drop_duplicates:')
print('  Sebelum : {:,} baris'.format(n_before_ds1))
print('  Sesudah : {:,} baris'.format(len(df1)))
print('  Dihapus : {:,} baris ({:.4f}%)'.format(n_dropped_ds1, n_dropped_ds1/n_before_ds1*100))

DS1 drop_duplicates:
  Sebelum : 100,000 baris
  Sesudah : 100,000 baris
  Dihapus : 0 baris (0.0000%)


In [36]:
# Parse Timestamp DS1
# DS1 (IBM Watson) TIDAK memiliki kolom timestamp — hanya daysOpen (integer).
# Kolom openedAt/resolvedAt/closedAt hanya ada di UCI Incident Log, bukan dataset ini.
print('DS1: Tidak ada kolom timestamp untuk dikonversi.')
print('  daysOpen dtype:', df1['daysOpen'].dtype)
neg_days = (df1['daysOpen'] < 0).sum()
print('  Nilai negatif daysOpen: {} (seharusnya 0 — konfirmasi tidak ada impossible value)'.format(neg_days))

DS1: Tidak ada kolom timestamp untuk dikonversi.
  daysOpen dtype: int64
  Nilai negatif daysOpen: 0 (seharusnya 0 — konfirmasi tidak ada impossible value)


In [37]:
# Lowercase + Strip Kolom Kategorikal DS1
# FiledAgainst, TicketType: plain text -> langsung lowercase + strip
# Kolom format 'N - Label' (Severity, Priority, RequestorSeniority, Satisfaction):
#   -> parsing integer N dan label string dilakukan (Feature Engineering)

PLAIN_CAT_DS1 = ['FiledAgainst', 'TicketType']
print('Lowercase + Strip DS1:')
for col in PLAIN_CAT_DS1:
    before_u = df1[col].nunique()
    df1[col] = df1[col].str.lower().str.strip()
    after_u  = df1[col].nunique()
    print('  {:20s}: {} unique -> {} unique. Values: {}'.format(
        col, before_u, after_u, sorted(df1[col].unique().tolist())))
print()
print('Kolom format N-Label (Severity, Priority, dll.) -> dihandle.')

Lowercase + Strip DS1:
  FiledAgainst        : 4 unique -> 4 unique. Values: ['access/login', 'hardware', 'software', 'systems']
  TicketType          : 2 unique -> 2 unique. Values: ['issue', 'request']

Kolom format N-Label (Severity, Priority, dll.) -> dihandle.


### 4.2 DS2 - Pembersihan Awal

Tahap ini menerapkan pembersihan dasar yang sama pada dataset pendukung (DS2): menghapus duplikat berdasarkan ID tiket unik dan menyeragamkan format teks pada kolom-kolom kategorikal.

In [38]:
# Verifikasi Primary Key Relasi Antar File DS2
# Kunci relasi: issues.id <-> snapshot.id (1:N) <-> history.issueid (1:N) <-> scored.id (subset)

issues_ids   = set(df_issues['id'].dropna().astype(int).unique())
snapshot_ids = set(df_snapshot['id'].dropna().astype(int).unique())
history_ids  = set(df_history['issueid'].dropna().astype(int).unique())
scored_id_col = 'id' if 'id' in df_scored.columns else df_scored.columns[0]
scored_ids   = set(df_scored[scored_id_col].dropna().astype(int).unique())
utr_id_col   = 'issuedid' if 'issuedid' in df_utterances.columns else df_utterances.columns[0]
utr_ids      = set(df_utterances[utr_id_col].dropna().astype(int).unique())

print('=== Primary Key Coverage ===')
print('  issues.csv         : {:,} unique id'.format(len(issues_ids)))
print('  issues_snapshot    : {:,} unique id'.format(len(snapshot_ids)))
print('  issues_history     : {:,} unique issueid'.format(len(history_ids)))
print('  scored sample      : {:,} unique id'.format(len(scored_ids)))
print('  utterances         : {:,} unique id'.format(len(utr_ids)))
print()
print('=== Subset Check (FK -> PK) ===')
print('  snapshot subset of issues : {}'.format(snapshot_ids.issubset(issues_ids)))
print('  history  subset of issues : {}'.format(history_ids.issubset(issues_ids)))
print('  scored   subset of issues : {}'.format(scored_ids.issubset(issues_ids)))
print('  utr      subset of issues : {}'.format(utr_ids.issubset(issues_ids)))
orphan_snap = snapshot_ids - issues_ids
orphan_hist = history_ids  - issues_ids
if orphan_snap: print('  PERINGATAN orphan snapshot: {:,}'.format(len(orphan_snap)))
if orphan_hist: print('  PERINGATAN orphan history : {:,}'.format(len(orphan_hist)))
print()
print('Konklusi: relasi antar file dikonfirmasi. PK valid.')

=== Primary Key Coverage ===
  issues.csv         : 66,691 unique id
  issues_snapshot    : 66,691 unique id
  issues_history     : 66,650 unique issueid
  scored sample      : 360 unique id
  utterances         : 360 unique id

=== Subset Check (FK -> PK) ===
  snapshot subset of issues : True
  history  subset of issues : True
  scored   subset of issues : True
  utr      subset of issues : True

Konklusi: relasi antar file dikonfirmasi. PK valid.


In [39]:
# Drop Duplikat DS2 issues.csv (berdasarkan ticket ID)
n_before_iss = len(df_issues)
dup_by_id    = df_issues.duplicated(subset=['id']).sum()
df_issues    = df_issues.drop_duplicates(subset=['id'], keep='first').reset_index(drop=True)
n_after_iss  = len(df_issues)

print('DS2 issues.csv drop_duplicates by id:')
print('  Sebelum : {:,} baris'.format(n_before_iss))
print('  Sesudah : {:,} baris'.format(n_after_iss))
print('  Dihapus : {:,} baris'.format(n_before_iss - n_after_iss))
print()
print('DS2 snapshot: TIDAK di-deduplicate (multi-row adalah desain per-assignee/per-turn)')
print('DS2 history : TIDAK di-deduplicate (setiap baris = satu event perubahan)')

DS2 issues.csv drop_duplicates by id:
  Sebelum : 66,691 baris
  Sesudah : 66,691 baris
  Dihapus : 0 baris

DS2 snapshot: TIDAK di-deduplicate (multi-row adalah desain per-assignee/per-turn)
DS2 history : TIDAK di-deduplicate (setiap baris = satu event perubahan)


In [40]:
# Lowercase + Strip Kolom Kategorikal DS2 issues.csv
CAT_SCRUB_DS2 = ['issue_priority', 'issue_type', 'issue_status', 'issue_resolution']
print('Lowercase + Strip DS2 issues.csv:')
for col in CAT_SCRUB_DS2:
    before_u = df_issues[col].nunique(dropna=False)
    df_issues[col] = df_issues[col].str.lower().str.strip()
    after_u  = df_issues[col].nunique(dropna=False)
    top_vals = df_issues[col].value_counts(dropna=False).head(4).index.tolist()
    print('  {:<30}: {} -> {} unique. Top: {}'.format(col, before_u, after_u, top_vals))
print()
if 'field' in df_history.columns:
    df_history['field'] = df_history['field'].str.lower().str.strip()
    print('issues_change_history field lowercase + strip: OK')
    print('  Unique field values:', df_history['field'].unique().tolist())

Lowercase + Strip DS2 issues.csv:
  issue_priority                : 7 -> 7 unique. Top: ['unknown', 'medium', 'high', 'highest']


  issue_type                    : 15 -> 15 unique. Top: ['ticket', 'service', 'subtask', 'story']

  issue_status                  : 15 -> 15 unique. Top: ['closed', 'done', 'waiting', 'open']


  issue_resolution              : 5 -> 5 unique. Top: ['done', "won't do", nan, 'duplicate']



issues_change_history field lowercase + strip: OK
  Unique field values: ['status', 'assignee']


## 5. Tahap Pembersihan 2: Penanganan Masalah Spesifik

Tahap ini menangani masalah yang lebih spesifik berdasarkan temuan diagnostik: nilai kosong, konversi satuan waktu, pembersihan teks, dan penanganan outlier. Setiap langkah menyebutkan nomor temuan dari bagian diagnostik sebagai justifikasi keputusan.

### 5.1 Penanganan Nilai Kosong

Setiap kolom dengan nilai kosong ditangani secara terpisah dengan strategi yang sesuai dengan mekanisme penyebabnya. Format keputusan yang digunakan:

`Kolom [nama]: [persentase]% kosong | Mekanisme: [MCAR/MAR/MNAR] - [bukti pendukung] | Keputusan | Alasan`

In [41]:
# 3A — DS1: Konfirmasi 0 null (Temuan 1B)
miss1_check = df1.isnull().sum().sum()
print('3A DS1 — Total null {}'.format(miss1_check))
if miss1_check == 0:
    print('Konfirmasi: DS1 tidak ada missing values. Tidak ada tindakan diperlukan.')
    print('(Simulasi IBM Watson sudah lengkap by design)')

3A DS1 — Total null 0
Konfirmasi: DS1 tidak ada missing values. Tidak ada tindakan diperlukan.
(Simulasi IBM Watson sudah lengkap by design)


In [42]:
# 3A — DS2 issue_assignee: Fill 'unassigned'
"""
Kolom issue_assignee: ~46.6% null
Mekanisme (dari 1B): MAR — null berkorelasi dengan tiket belum di-assign
  (tiket baru masuk / status open tidak punya assignee by design)
Keputusan: Fill sentinel 'unassigned'
Alasan: null bermakna 'belum ada assignee', bukan data hilang;
        sentinel 'unassigned' lebih informatif dari NaN untuk analisis beban kerja
"""

n_null_ass = df_issues['issue_assignee'].isnull().sum()
df_issues['issue_assignee'] = df_issues['issue_assignee'].fillna('unassigned')
print('issue_assignee: {:,} null ({:.1f}%) -> fill "unassigned"'.format(
    n_null_ass, n_null_ass/len(df_issues)*100))
print('  Verifikasi null sesudah:', df_issues['issue_assignee'].isnull().sum())

issue_assignee: 31,047 null (46.6%) -> fill "unassigned"
  Verifikasi null sesudah: 0


In [43]:
# 3A — DS2 isResolved Flag + issue_resolution
"""
Kolom issue_resolution_date: ~1.3% null
Mekanisme (dari 1B): MAR — null berkorelasi status Open/In Progress
  (cross-tab 1B: closed/done hampir selalu punya resolution_date)
Keputusan: Buat flag isResolved (1=resolved, 0=open); PERTAHANKAN null tanggal
Alasan: null bukan data hilang — tiket memang belum selesai;
        mengisi tanggal palsu merusak analisis durasi resolusi

Kolom issue_resolution: ~1.3% null (satu paket dengan issue_resolution_date)
Mekanisme: MAR (sama)
Keputusan: Fill 'unknown'
Alasan: konsisten dengan penanganan issue_assignee
"""

n_null_resdate = df_issues['issue_resolution_date'].isnull().sum()
n_null_res     = df_issues['issue_resolution'].isnull().sum()

df_issues['isResolved'] = df_issues['issue_resolution_date'].notna().astype(int)
df_issues['issue_resolution'] = df_issues['issue_resolution'].fillna('unknown')

print('issue_resolution_date: {:,} null ({:.1f}%) -> DIPERTAHANKAN; flag isResolved dibuat'.format(
    n_null_resdate, n_null_resdate/len(df_issues)*100))
print('issue_resolution     : {:,} null ({:.1f}%) -> fill "unknown"'.format(
    n_null_res, n_null_res/len(df_issues)*100))
print()
print('isResolved distribution:')
vc_resolved = df_issues['isResolved'].value_counts()
for v, c in vc_resolved.items():
    lbl = 'resolved' if v == 1 else 'belum resolved'
    print('  {} ({}): {:,} tiket ({:.1f}%)'.format(v, lbl, c, c/len(df_issues)*100))

issue_resolution_date: 853 null (1.3%) -> DIPERTAHANKAN; flag isResolved dibuat


issue_resolution     : 853 null (1.3%) -> fill "unknown"

isResolved distribution:
  1 (resolved): 65,838 tiket (98.7%)
  0 (belum resolved): 853 tiket (1.3%)


In [44]:
# 3A — DS2 Drop 16 wf_* Kolom + Fill 4 wf_* Retained
"""
16 kolom wf_* (95-100% null):
Mekanisme (dari 1B): MNAR by design — null = tiket tidak pernah lewat state itu
Keputusan: DROP — tidak ada informasi bermakna di 95-100% missing
Alasan: 16 kolom terlalu banyak noise; wfe_* (0% missing) tetap dipertahankan

wf_resolved (~57.8%), wf_in_progress (~44.9%), wf_waiting (~70.5%), wf_open (~0.1%):
Mekanisme: MNAR — null = 0 detik di state tersebut (tidak pernah memasuki state)
Keputusan: Fill 0
Alasan: null secara semantik adalah 0 detik; fill 0 mempertahankan informasi temporal
"""

WF_TO_DROP = [
    'wf_in_review', 'wf_rejected', 'wf_deployment', 'wf_cancelled',
    'wf_pending_customer_approval', 'wf_testing_monitoring', 'wf_monitoring',
    'wf_to_do', 'wf_done', 'wf_pending_deployment', 'wf_closed',
    'wf_approved', 'wf_under_review', 'wf_resolved_under_monitoring',
    'wf_reopened', 'wf_validation'
]
WF_FILL_ZERO = ['wf_resolved', 'wf_in_progress', 'wf_waiting', 'wf_open']

drop_exist = [c for c in WF_TO_DROP if c in df_issues.columns]
df_issues.drop(columns=drop_exist, inplace=True)
print('Kolom di-drop (issues.csv): {} kolom'.format(len(drop_exist)))
print('  Kolom:', drop_exist[:8], '...' if len(drop_exist) > 8 else '')
print()
for col in WF_FILL_ZERO:
    if col in df_issues.columns:
        n_null = df_issues[col].isnull().sum()
        df_issues[col] = df_issues[col].fillna(0)
        print('  {:30s}: {:,} null -> fill 0'.format(col, n_null))
print()
print('DS2 issues.csv shape setelah 3A:', df_issues.shape)

Kolom di-drop (issues.csv): 16 kolom
  Kolom: ['wf_in_review', 'wf_rejected', 'wf_deployment', 'wf_cancelled', 'wf_pending_customer_approval', 'wf_testing_monitoring', 'wf_monitoring', 'wf_to_do'] ...

  wf_resolved                   : 38,546 null -> fill 0
  wf_in_progress                : 29,944 null -> fill 0
  wf_waiting                    : 46,987 null -> fill 0
  wf_open                       : 78 null -> fill 0

DS2 issues.csv shape setelah 3A: (66691, 43)


In [45]:
# 3A — DS2 Snapshot: Terapkan Handling yang Sama
for col in ['issue_priority', 'issue_type', 'issue_status', 'issue_resolution']:
    if col in df_snapshot.columns:
        df_snapshot[col] = df_snapshot[col].str.lower().str.strip()
if 'issue_assignee'       in df_snapshot.columns:
    df_snapshot['issue_assignee'] = df_snapshot['issue_assignee'].fillna('unassigned')
if 'issue_resolution_date' in df_snapshot.columns:
    df_snapshot['isResolved'] = df_snapshot['issue_resolution_date'].notna().astype(int)
if 'issue_resolution'      in df_snapshot.columns:
    df_snapshot['issue_resolution'] = df_snapshot['issue_resolution'].fillna('unknown')

drop_snap = [c for c in WF_TO_DROP if c in df_snapshot.columns]
df_snapshot.drop(columns=drop_snap, inplace=True)
for col in WF_FILL_ZERO:
    if col in df_snapshot.columns:
        df_snapshot[col] = df_snapshot[col].fillna(0)
print('Snapshot 3A selesai. Shape:', df_snapshot.shape)

Snapshot 3A selesai. Shape: (90963, 45)


### 5.2 Menghitung Durasi Resolusi DS1

Rencana awal menggunakan rumus `resolutionDurationHours = (resolvedAt - openedAt)` dalam satuan jam.

**Penyesuaian untuk IBM Watson:** Dataset ini tidak memiliki kolom timestamp `resolvedAt` dan `openedAt`. Sebagai gantinya, kami menggunakan kolom `daysOpen` yang sudah tersedia sebagai proksi durasi resolusi. Kolom baru `resolutionDurationDays` dibuat langsung dari `daysOpen`.

In [46]:
# 3B — Durasi Resolusi DS1 (Adaptasi — tidak ada timestamp)
"""
PREPROCESSING_PLAN: resolutionDurationHours = resolvedAt - openedAt
Temuan 1D: DS1 IBM Watson tidak punya kolom timestamp (berbeda dari UCI Incident Log)
Keputusan: buat resolutionDurationDays = daysOpen (proxy)
Alasan: daysOpen merepresentasikan berapa hari tiket terbuka — semantik serupa
"""

df1['resolutionDurationDays'] = df1['daysOpen'].copy()
# Catatan: # KPI durasi resolusi DS1 — proxy karena tidak ada timestamp
print('3B DS1 — resolutionDurationDays = daysOpen (proxy, tidak ada timestamp).')
print(df1['resolutionDurationDays'].describe().round(2))
print()
print('Catatan untuk Modeler: DS1 tidak punya isResolved flag (tidak ada resolvedAt).')
print('Semua tiket IBM Watson diasumsikan closed (dataset tidak menyimpan tiket open).')

3B DS1 — resolutionDurationDays = daysOpen (proxy, tidak ada timestamp).
count   100000.00
mean         6.84
std          7.38
min          0.00
25%          1.00
50%          5.00
75%         10.00
max         54.00
Name: resolutionDurationDays, dtype: float64

Catatan untuk Modeler: DS1 tidak punya isResolved flag (tidak ada resolvedAt).
Semua tiket IBM Watson diasumsikan closed (dataset tidak menyimpan tiket open).


### 5.3 Konversi Format Waktu DS2

Kolom-kolom tanggal dan waktu di DS2 masih tersimpan sebagai teks (string). Bagian ini mengubahnya menjadi tipe datetime yang sesungguhnya agar bisa dilakukan operasi perhitungan waktu seperti selisih tanggal dan ekstraksi bulan atau tahun.

In [47]:
# 3C — Konversi Timestamp DS2 ke datetime
# temuan 1D: semua timestamp masih object dtype
# Strategi: pd.to_datetime(format='ISO8601', utc=True, errors='coerce')

TS_COLS = ['started', 'ended', 'issue_created', 'issue_resolution_date', 'last_change_date']
print('Konversi timestamp DS2 issues.csv:')
for col in TS_COLS:
    if col not in df_issues.columns:
        continue
    n_null_before = df_issues[col].isnull().sum()
    df_issues[col + '_dt'] = pd.to_datetime(
        df_issues[col], format='ISO8601', utc=True, errors='coerce')
    n_null_after = df_issues[col + '_dt'].isnull().sum()
    n_parse_err  = max(0, n_null_after - n_null_before)
    print('  {:<35}: OK | parse errors: {}'.format(col + '_dt', n_parse_err))

if 'created' in df_history.columns:
    df_history['created_dt'] = pd.to_datetime(
        df_history['created'], format='ISO8601', utc=True, errors='coerce')
    print('  history.created_dt: OK')
if 'created' in df_utterances.columns:
    df_utterances['created_dt'] = pd.to_datetime(
        df_utterances['created'], format='ISO8601', utc=True, errors='coerce')
    print('  utterances.created_dt: OK')

Konversi timestamp DS2 issues.csv:
  started_dt                         : OK | parse errors: 0


  ended_dt                           : OK | parse errors: 0


  issue_created_dt                   : OK | parse errors: 0


  issue_resolution_date_dt           : OK | parse errors: 0


  last_change_date_dt                : OK | parse errors: 0


  history.created_dt: OK
  utterances.created_dt: OK


In [48]:
# 3C — Kolom Waktu Turunan DS2
# PREPROCESSING_PLAN: timePerStep -> timePerStepHours (bagi 3600)
# Kolom aktual: wf_total_time (total waktu DETIK, per FEATURES.md)

# totalTimeHours = wf_total_time / 3600
# Alasan: # Satuan jam lebih intuitif dari detik untuk analisis SLA
df_issues['totalTimeHours'] = df_issues['wf_total_time'] / 3600
print('totalTimeHours = wf_total_time / 3600')
print(df_issues['totalTimeHours'].describe().round(2))
print()

# timePerStepHours = wf_total_time / processing_steps / 3600
# Alasan: # Efisiensi rata-rata per langkah workflow
mask_vs = df_issues['processing_steps'] > 0
df_issues['timePerStepHours'] = np.nan
df_issues.loc[mask_vs, 'timePerStepHours'] = (
    df_issues.loc[mask_vs, 'wf_total_time']
    / df_issues.loc[mask_vs, 'processing_steps'] / 3600
)
print('timePerStepHours = wf_total_time / processing_steps / 3600')
print('  Terhitung: {:,} tiket (processing_steps > 0)'.format(mask_vs.sum()))
print()

# resolutionDurationHours = (issue_resolution_date - issue_created) / 3600
# Alasan: # KPI utama SLA — durasi sebenarnya dari buat hingga selesai
if 'issue_resolution_date_dt' in df_issues.columns and 'issue_created_dt' in df_issues.columns:
    df_issues['resolutionDurationHours'] = (
        (df_issues['issue_resolution_date_dt'] - df_issues['issue_created_dt']).dt.total_seconds() / 3600
    )
    n_rh = df_issues['resolutionDurationHours'].notna().sum()
    print('resolutionDurationHours = (resolution_date - created) / 3600')
    print('  Terhitung: {:,} tiket (isResolved=1)'.format(n_rh))
    print(df_issues['resolutionDurationHours'].describe().round(2))

totalTimeHours = wf_total_time / 3600
count   66691.00
mean    18293.46
std     27698.71
min         0.00
25%       122.46
50%       822.77
75%     35921.94
max     99220.54
Name: totalTimeHours, dtype: float64

timePerStepHours = wf_total_time / processing_steps / 3600
  Terhitung: 66,613 tiket (processing_steps > 0)

resolutionDurationHours = (resolution_date - created) / 3600
  Terhitung: 65,838 tiket (isResolved=1)
count   65838.00
mean    18482.55
std     27805.94
min         0.00
25%       124.32
50%       837.81
75%     36508.15
max     99220.54
Name: resolutionDurationHours, dtype: float64


### 5.4 Pembersihan Teks Utterances

Dataset utterances berisi percakapan antara pengguna dan agen helpdesk dalam bentuk teks bebas. Teks ini perlu dibersihkan sebelum bisa dianalisis menggunakan teknik NLP. Langkah-langkah pembersihan meliputi: mengubah semua teks ke huruf kecil, menghapus karakter selain huruf dan angka, dan mengganti nama orang atau tempat dengan token standar.

In [49]:
# 3D — Pipeline Cleaning Teks Utterances
# Pipeline: lowercase -> hapus non-alfabet -> normalisasi whitespace

import re

def cleanText(text):
    # Step 1: lowercase
    if pd.isna(text) or str(text).strip() == '':
        return ''
    text = str(text).lower()
    # Step 2: hapus karakter non-alfabet (pertahankan spasi)
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Step 3: strip whitespace berlebih
    text = ' '.join(text.split())
    return text

# Identifikasi kolom teks (FEATURES.md: 'actionbody')
text_col = 'actionbody' if 'actionbody' in df_utterances.columns else df_utterances.columns[-1]
print('Kolom teks:', text_col)
print('Sample teks asli:')
for i, t in enumerate(df_utterances[text_col].head(3).tolist()):
    print('  [{}] {}'.format(i, str(t)[:80]))

df_utterances['messageClean'] = df_utterances[text_col].apply(cleanText)
df_utterances['wordCount']    = df_utterances['messageClean'].apply(
    lambda x: len(x.split()) if x.strip() else 0)

print()
n_short = (df_utterances['wordCount'] < 3).sum()
print('Baris wordCount < 3: {:,} ({:.2f}%)'.format(n_short, n_short/len(df_utterances)*100))
n_before_utr = len(df_utterances)
df_utterances = df_utterances[df_utterances['wordCount'] >= 3].copy().reset_index(drop=True)
print('Drop wordCount < 3: {:,} -> {:,} baris ({:,} dihapus)'.format(
    n_before_utr, len(df_utterances), n_before_utr - len(df_utterances)))
print()
print('=== wordCount stats setelah filter ===')
print(df_utterances['wordCount'].describe().round(2))

Kolom teks: actionbody
Sample teks asli:
  [0] dear ph_name team
  [1] we need your urgent support to fix list of vulnerabilities reported in ph_name a
  [2] please provide resolution date before end of ‘january ph_technical .



Baris wordCount < 3: 15,649 (51.98%)


Drop wordCount < 3: 30,104 -> 14,455 baris (15,649 dihapus)

=== wordCount stats setelah filter ===
count   14455.00
mean       13.25
std        12.58
min         3.00
25%         5.00
50%        10.00
75%        16.00
max       142.00
Name: wordCount, dtype: float64


### 5.5 Penanganan Outlier

Berdasarkan hasil diagnostik, outlier pada `daysOpen` (DS1) dan `wf_total_time` (DS2) adalah tiket yang memang benar-benar kompleks atau berumur sangat lama, bukan kesalahan pencatatan data. Oleh karena itu strategi yang digunakan adalah:
- **Cap di persentil ke-99:** nilai ekstrem dibatasi pada batas atas P99 agar tidak mendistorsi statistik, namun tiket tersebut tetap ada dalam dataset
- **Flag tiket panjang:** kolom penanda `isLongTicket` dibuat untuk menandai tiket dengan durasi di atas P99

In [50]:
# 3E — DS1: Cap daysOpen P99 + Flag isLongTicket
# Temuan 1E: daysOpen outlier genuine extreme, tidak ada nilai negatif
# Keputusan: cap P99 untuk mengurangi pengaruh extreme outlier
# isLongTicket: # Flag tiket yang sangat lama terbuka — untuk SLA risk analysis

p95_days = float(df1_orig['daysOpen'].quantile(0.95))
p99_days = float(df1_orig['daysOpen'].quantile(0.99))

# Flag SEBELUM capping (gunakan nilai asli)
df1['isLongTicket'] = (df1['daysOpen'] > p95_days).astype(int)

n_capped_days = int((df1['daysOpen'] > p99_days).sum())
df1['daysOpen'] = df1['daysOpen'].clip(upper=p99_days)
df1['resolutionDurationDays'] = df1['daysOpen'].copy()

print('DS1 daysOpen outlier handling:')
print('  P95: {:.0f} hari (threshold isLongTicket)'.format(p95_days))
print('  P99: {:.0f} hari (cap value)'.format(p99_days))
print('  isLongTicket (> P95): {:,} tiket ({:.2f}%)'.format(
    df1['isLongTicket'].sum(), df1['isLongTicket'].mean()*100))
print('  Dicap P99            : {:,} tiket'.format(n_capped_days))
print('  daysOpen max setelah cap: {:.0f} hari'.format(df1['daysOpen'].max()))

DS1 daysOpen outlier handling:
  P95: 22 hari (threshold isLongTicket)
  P99: 31 hari (cap value)
  isLongTicket (> P95): 4,499 tiket (4.50%)
  Dicap P99            : 988 tiket
  daysOpen max setelah cap: 31 hari


In [51]:
# 3E — DS2: Cap P99 Outlier Numerik Utama
# Temuan 1E: wf_total_time, processing_steps, resolutionDurationHours genuine extreme
# Keputusan: cap P99 — tidak set NaN (tidak ada impossible value negatif terbukti dari 1E)

# Cap wf_total_time
p99_wt = float(df_issues_orig['wf_total_time'].quantile(0.99))
n_cap_wt = int((df_issues['wf_total_time'] > p99_wt).sum())
df_issues['wf_total_time'] = df_issues['wf_total_time'].clip(upper=p99_wt)
df_issues['totalTimeHours'] = df_issues['wf_total_time'] / 3600
print('wf_total_time cap P99: {:.1f} jam | dicap: {:,} tiket'.format(p99_wt/3600, n_cap_wt))

# Cap processing_steps
p99_ps = float(df_issues_orig['processing_steps'].quantile(0.99))
n_cap_ps = int((df_issues['processing_steps'] > p99_ps).sum())
df_issues['processing_steps'] = df_issues['processing_steps'].clip(upper=p99_ps)
print('processing_steps cap P99: {:.0f} | dicap: {:,} tiket'.format(p99_ps, n_cap_ps))

# Cap resolutionDurationHours
if 'resolutionDurationHours' in df_issues.columns:
    valid_rd = df_issues['resolutionDurationHours'].dropna()
    if len(valid_rd) > 0:
        p99_rd   = float(valid_rd.quantile(0.99))
        n_cap_rd = int((df_issues['resolutionDurationHours'] > p99_rd).sum())
        df_issues['resolutionDurationHours'] = df_issues['resolutionDurationHours'].clip(upper=p99_rd)
        print('resolutionDurationHours cap P99: {:.1f} jam | dicap: {:,} tiket'.format(p99_rd, n_cap_rd))

# Recalculate timePerStepHours setelah caps
mask_vs2 = df_issues['processing_steps'] > 0
df_issues['timePerStepHours'] = np.nan
df_issues.loc[mask_vs2, 'timePerStepHours'] = (
    df_issues.loc[mask_vs2, 'wf_total_time']
    / df_issues.loc[mask_vs2, 'processing_steps'] / 3600
)
print('timePerStepHours recalculated setelah caps.')

wf_total_time cap P99: 90910.6 jam | dicap: 667 tiket
processing_steps cap P99: 13 | dicap: 664 tiket
resolutionDurationHours cap P99: 90929.0 jam | dicap: 659 tiket
timePerStepHours recalculated setelah caps.


In [52]:
# 3E — DS2: Flag isComplex
# Temuan 1E: wfe_reopened outlier = tiket dibuka ulang (operasional kompleks)
#            issue_contr_count outlier = banyak kontributor (proxy reassignment)
#
# Ekuivalen DS2 untuk reassignmentCount/reopenCount di UCI:
#   wfe_reopened > 0 (any reopen) ATAU issue_contr_count > median
# Alasan: # Flag tiket kompleks — sesuai pertanyaan analitik kelompok

thresh_reopen = 0
thresh_contr  = float(df_issues_orig['issue_contr_count'].median())

df_issues['isComplex'] = (
    (df_issues['wfe_reopened'] > thresh_reopen) |
    (df_issues['issue_contr_count'] > thresh_contr)
).astype(int)

print('isComplex thresholds:')
print('  wfe_reopened    > {} (any reopen)'.format(thresh_reopen))
print('  issue_contr_count > {:.1f} (median)'.format(thresh_contr))
print()
print('isComplex distribution DS2:')
vc_cmx = df_issues['isComplex'].value_counts()
for v, c in vc_cmx.sort_index().items():
    lbl = 'Complex' if v == 1 else 'Normal'
    print('  {} ({}): {:,} tiket ({:.2f}%)'.format(v, lbl, c, c/len(df_issues)*100))

isComplex thresholds:
  wfe_reopened    > 0 (any reopen)
  issue_contr_count > 1.0 (median)

isComplex distribution DS2:
  0 (Normal): 53,441 tiket (80.13%)
  1 (Complex): 13,250 tiket (19.87%)


### 5.6 Sampling

Bagian ini mengevaluasi apakah diperlukan pengambilan sampel (sampling) pada dataset ini dan mencatat keputusan yang diambil beserta alasannya.

In [53]:
# 3F — Keputusan Sampling (berdasarkan temuan 1F)
# Aturan: SMOTE hanya jika rasio > 10:1 DAN label sebagai target klasifikasi
#         SMOTE hanya pada training split, TIDAK seluruh dataset

sev_vc    = df1['Severity'].value_counts()
sev_ratio = sev_vc.max() / sev_vc.min()
print('DS1 Severity ratio (mayoritas/minoritas): {:.1f}:1'.format(sev_ratio))
print(sev_vc.to_string())
print()
pri_no_unk = df_issues[df_issues['issue_priority'] != 'unknown']['issue_priority'].value_counts()
if len(pri_no_unk) > 1:
    pri_ratio = pri_no_unk.max() / pri_no_unk.min()
    print('DS2 priority ratio (tanpa unknown): {:.1f}:1'.format(pri_ratio))
    print(pri_no_unk.to_string())
print()
print('=== KEPUTUSAN FINAL SAMPLING ===')
print('SMOTE TIDAK dieksekusi di Fase Scrub.')
print('Alasan:')
print('  1. SMOTE hanya pada training split (scope Modeler/Daffa)')
print('  2. DS2 priority perlu normalisasi dulu  sebelum evaluasi')
print('  3. Untuk EDA & Explore: tidak diperlukan sampling')
print('  4. Dokumen handoff: class_weight atau SMOTE di X_train')

DS1 Severity ratio (mayoritas/minoritas): 247.7:1
Severity
2 - Normal          90912
3 - Major            4974
1 - Minor            2317
4 - Critical         1430
0 - Unclassified      367

DS2 priority ratio (tanpa unknown): 295.1:1
issue_priority
medium     24788
high        4554
highest     2084
blocker      656
low          560
lowest        84

=== KEPUTUSAN FINAL SAMPLING ===
SMOTE TIDAK dieksekusi di Fase Scrub.
Alasan:
  1. SMOTE hanya pada training split (scope Modeler/Daffa)
  2. DS2 priority perlu normalisasi dulu  sebelum evaluasi
  3. Untuk EDA & Explore: tidak diperlukan sampling
  4. Dokumen handoff: class_weight atau SMOTE di X_train


## 6. Rekayasa Fitur (Feature Engineering)

Tahap ini membuat kolom-kolom baru yang tidak ada di data asli tetapi dibutuhkan untuk analisis dan pemodelan. Setiap fitur baru disertai alasan pembuatannya di dalam komentar kode.

Fitur yang ditandai **[ADAPTASI]** berarti fitur tersebut merupakan penyesuaian dari rencana awal karena DS1 menggunakan format IBM Watson, bukan dataset UCI yang direncanakan semula.

### 6.1 DS1 - Rekayasa Fitur

Kolom-kolom DS1 yang tersedia untuk pembuatan fitur baru adalah: `Severity`, `Priority`, `RequestorSeniority`, dan `Satisfaction` yang semuanya dalam format `N - Label` (angka diikuti tanda hubung dan label teks), serta `FiledAgainst`, `TicketType`, `daysOpen`, dan `ITOwner`.

Fitur-fitur baru yang akan dibuat antara lain:
- Representasi numerik dari setiap kolom kategorikal
- Flag biner untuk kondisi tertentu seperti prioritas tinggi dan tiket yang sudah lama terbuka
- Indikator konsistensi antara nilai severity dan priority

In [54]:
# DS1: Parse Kolom Format 'N - Label'
# [ADAPTASI]: DS1 IBM Watson menyimpan Severity/Priority/dll sebagai 'N - LabelName'
# -> Parse integer N (ordinal) dan string label (kategorikal) terpisah

import re as _re

def parseNLabel(series):
    # Alasan: # Ordinal numerik diperlukan untuk statistik; label untuk visualisasi
    level = series.str.extract(r'^(\d+)')[0].astype(float)
    label = series.str.replace(r'^\d+\s*-\s*', '', regex=True).str.strip().str.lower()
    return level, label

# Severity: e.g. '1 - Minor', '2 - Normal', '3 - Major', '4 - Critical'
df1['severityLevel'], df1['severityLabel'] = parseNLabel(df1['Severity'])
# Alasan: # Ordinal severity diperlukan untuk ordering dan model; label untuk viz

# Priority: e.g. '1 - Unassigned', '2 - Low', '3 - Medium', '4 - High'
df1['priorityLevel'], df1['priorityLabel'] = parseNLabel(df1['Priority'])
# Alasan: # Ordinal priority untuk filter high priority dan model klasifikasi

# RequestorSeniority: e.g. '1 - Junior', '2 - Mid-Level', '3 - Senior', '4 - Expert'
df1['seniorityLevel'], df1['seniorityLabel'] = parseNLabel(df1['RequestorSeniority'])
# Alasan: # Hipotesis: seniority berkorelasi dengan urgency dan prioritas penyelesaian

# Satisfaction: e.g. '0 - Unknown', '1 - Very Unsatisfied',... '5 - Very Satisfied'
df1['satisfactionLevel'], df1['satisfactionLabel'] = parseNLabel(df1['Satisfaction'])
# Unknown/N/A -> NaN sudah dihandle oleh.astype(float) untuk non-numeric strings
# Alasan: # Target KPI kepuasan pelanggan — numerik untuk regresi, label untuk viz

print('=== Feature Baru DS1 — Parse N-Label ===')
for orig, lvl, lbl in [
    ('Severity',           'severityLevel',     'severityLabel'),
    ('Priority',           'priorityLevel',     'priorityLabel'),
    ('RequestorSeniority', 'seniorityLevel',    'seniorityLabel'),
    ('Satisfaction',       'satisfactionLevel', 'satisfactionLabel'),
]:
    lev_vals = sorted([v for v in df1[lvl].dropna().unique()])
    lbl_vals = sorted([v for v in df1[lbl].dropna().unique()])
    print('  {} -> {} levels: {}'.format(orig, len(lev_vals), lev_vals))
    print('         labels: {}'.format(lbl_vals))

=== Feature Baru DS1 — Parse N-Label ===
  Severity -> 5 levels: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
         labels: ['critical', 'major', 'minor', 'normal', 'unclassified']
  Priority -> 4 levels: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0)]
         labels: ['high', 'low', 'medium', 'unassigned']
  RequestorSeniority -> 4 levels: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
         labels: ['junior', 'management', 'regular', 'senior']
  Satisfaction -> 4 levels: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0)]
         labels: ['highly satisfied', 'satisfied', 'unknown', 'unsatisfied']


In [55]:
# DS1: isHighPriority + priorityVerified
#
# [ADAPTASI] urgencyLabel, impactLabel, isResolved, slaMet:
#   TIDAK APPLICABLE — DS1 tidak punya kolom urgency, impact, resolvedAt, madeSla
#   (Kolom-kolom ini hanya ada di UCI Incident Log)
#
# Fitur yang bisa dibuat dari DS1:

# isHighPriority: 1 jika priorityLevel == nilai tertinggi yang tersedia
# Alasan: # Flag tiket prioritas tertinggi — untuk analisis SLA risk
max_priority = df1['priorityLevel'].max()
df1['isHighPriority'] = (df1['priorityLevel'] == max_priority).astype(int)
print('isHighPriority (priorityLevel == {:.0f}): {:,} tiket ({:.2f}%)'.format(
    max_priority, df1['isHighPriority'].sum(), df1['isHighPriority'].mean()*100))

# priorityVerified: cross-check apakah severity tinggi -> priority tinggi
# Alasan: # Detect redundancy — jika korelasi tinggi, salah satu bisa di-drop di model
corr_sev_pri = df1[['severityLevel', 'priorityLevel']].corr(method='spearman')
spearman_r   = corr_sev_pri.loc['severityLevel', 'priorityLevel']
print()
print('Spearman corr(severityLevel, priorityLevel): {:.4f}'.format(spearman_r))
if abs(spearman_r) > 0.8:
    print('  => Korelasi sangat tinggi (>0.8): priority kemungkinan redundan dengan severity')
elif abs(spearman_r) > 0.5:
    print('  => Korelasi moderat (0.5-0.8): ada variasi tambahan di priority')
else:
    print('  => Korelasi rendah (<0.5): severity dan priority independent')

print()
print('Cross-tab severityLevel x priorityLevel:')
ct_svp = pd.crosstab(df1['severityLevel'], df1['priorityLevel'])
display(ct_svp)

# priorityVerified flag: 1 jika severity dan priority konsisten arah
sev_med = df1['severityLevel'].median()
pri_med = df1['priorityLevel'].median()
df1['priorityVerified'] = (
    ((df1['severityLevel'] >= sev_med) & (df1['priorityLevel'] >= pri_med)) |
    ((df1['severityLevel'] <  sev_med) & (df1['priorityLevel'] <  pri_med))
).astype(int)
print('priorityVerified (arah konsisten): {:,} ({:.2f}%)'.format(
    df1['priorityVerified'].sum(), df1['priorityVerified'].mean()*100))

isHighPriority (priorityLevel == 3): 36,498 tiket (36.50%)

Spearman corr(severityLevel, priorityLevel): 0.0292
  => Korelasi rendah (<0.5): severity dan priority independent

Cross-tab severityLevel x priorityLevel:


priorityLevel,0.00,1.00,2.00,3.00
severityLevel,,,,
0.00,120,82,57,108
1.00,642,564,420,691
2.00,27478,15670,14834,32930
3.00,1469,626,740,2139
4.00,418,175,207,630


priorityVerified (arah konsisten): 52,888 (52.89%)


### 6.2 DS2 - Rekayasa Fitur

Fitur-fitur baru untuk DS2 dibuat berdasarkan kolom-kolom yang sudah ada, meliputi:
- Normalisasi nilai priority yang tidak konsisten (`priorityNormalized`)
- Pengelompokan kecepatan resolusi ke dalam tiga kategori: cepat, sedang, dan lambat (`resolutionSpeedCategory`)
- Skor performa gabungan dari beberapa kolom kualitas (`compositeScore`)
- Penghitungan jumlah utterance (pesan) per tiket (`utteranceCountPerIssue`)

In [56]:
# DS2: priorityNormalized — Normalisasi issue_priority
# Alasan: # 'unknown' mendominasi ~51% — diperlukan klasifikasi ulang untuk analisis
# Mapping berdasarkan FEATURES.md dan nilai aktual setelah lowercase

PRIORITY_MAP = {
    'blocker': 'high', 'highest': 'high', 'high': 'high',
    'medium': 'medium',
    'low': 'low', 'lowest': 'low',
    'unknown': 'unknown',
}
df_issues['priorityNormalized'] = (
    df_issues['issue_priority'].map(PRIORITY_MAP).fillna('unknown')
)
print('priorityNormalized distribution:')
vc_pri2 = df_issues['priorityNormalized'].value_counts(dropna=False)
for v, c in vc_pri2.items():
    print('  {:10s}: {:,} ({:.2f}%)'.format(str(v), c, c/len(df_issues)*100))

priorityNormalized distribution:
  unknown   : 33,965 (50.93%)
  medium    : 24,788 (37.17%)
  high      : 7,294 (10.94%)
  low       : 644 (0.97%)


In [57]:
# DS2: resolutionSpeedCategory — qcut 3 bin dari resolutionDurationHours
# Alasan: # qcut robust untuk distribusi skewed (vs cut yang menghasilkan bin tidak seimbang)
#          3 bin (fast/medium/slow) intuitif untuk analisis SLA

resolved_mask = df_issues['isResolved'] == 1
n_resolved    = resolved_mask.sum()
print('Tiket resolved untuk qcut: {:,}'.format(n_resolved))

df_issues['resolutionSpeedCategory'] = pd.Series([pd.NA] * len(df_issues), dtype=object)
if 'resolutionDurationHours' in df_issues.columns and n_resolved >= 50:
    cats, bins = pd.qcut(
        df_issues.loc[resolved_mask, 'resolutionDurationHours'],
        q=3, labels=['fast', 'medium', 'slow'],
        retbins=True, duplicates='drop'
    )
    df_issues.loc[resolved_mask, 'resolutionSpeedCategory'] = cats.astype(object).values
    print('Bin boundaries (jam): {}'.format([round(b, 1) for b in bins]))
    print()
    print('resolutionSpeedCategory (resolved only):')
    print(df_issues.loc[resolved_mask, 'resolutionSpeedCategory'].value_counts().to_string())
else:
    print('Skip: resolutionDurationHours tidak tersedia atau < 50 tiket resolved.')

Tiket resolved untuk qcut: 65,838
Bin boundaries (jam): [np.float64(0.0), np.float64(235.0), np.float64(11165.0), np.float64(90929.0)]

resolutionSpeedCategory (resolved only):
resolutionSpeedCategory
fast      21946
medium    21946
slow      21946


In [58]:
# DS2 Scored: compositeScore + performanceBinary
# Alasan: # Binary lebih stabil dari multi-kelas untuk klasifikasi pertama;
#          threshold >=4 standar Likert; compositeScore rata-rata Q lebih robust

q_cols = [c for c in ['Q1', 'Q2', 'Q3'] if c in df_scored.columns]
print('Q columns ditemukan:', q_cols)

if q_cols:
    df_scored['compositeScore'] = df_scored[q_cols].mean(axis=1)
    # Alasan: # Rata-rata Q lebih robust dari Q tunggal untuk mengukur kualitas keseluruhan

    df_scored['performanceBinary'] = np.where(
        df_scored['compositeScore'] >= 4.0, 'good', 'needs_improvement'
    )
    # Alasan: # Threshold 4/5 = 'good' sesuai standar Likert; label actionable untuk model

    vc_pb    = df_scored['performanceBinary'].value_counts()
    ratio_pb = vc_pb.max() / vc_pb.min() if vc_pb.min() > 0 else float('inf')
    print()
    print('performanceBinary distribution:')
    for v, c in vc_pb.items():
        print('  {:20s}: {:,} ({:.2f}%)'.format(v, c, c/len(df_scored)*100))
    print('Rasio mayoritas/minoritas: {:.1f}:1'.format(ratio_pb))
    if ratio_pb > 10:
        print('=> Imbalanced >10:1: SMOTE pada training split direkomendasikan ke Modeler')
    else:
        print('=> class_weight parameter sudah cukup (ratio <= 10:1)')

Q columns ditemukan: ['Q1', 'Q2', 'Q3']

performanceBinary distribution:
  good                : 439 (58.77%)
  needs_improvement   : 308 (41.23%)
Rasio mayoritas/minoritas: 1.4:1
=> class_weight parameter sudah cukup (ratio <= 10:1)


In [59]:
# DS2: utteranceCountPerIssue — Agregasi count per issue
# Alasan: # Jumlah utterance per tiket = proxy intensitas komunikasi/kompleksitas
# Lebih banyak percakapan -> tiket lebih kompleks atau membutuhkan banyak klarifikasi

print('Kolom ID di utterances:', utr_id_col)
utr_count = (df_utterances.groupby(utr_id_col).size().reset_index(name='utteranceCountPerIssue'))
print('Unique issues dengan utterances: {:,}'.format(len(utr_count)))
print(utr_count['utteranceCountPerIssue'].describe().round(2))

# Merge ke df_issues
df_issues = df_issues.merge(
    utr_count.rename(columns={utr_id_col: 'id'}),
    on='id', how='left'
)
df_issues['utteranceCountPerIssue'] = df_issues['utteranceCountPerIssue'].fillna(0).astype(int)
print()
print('utteranceCountPerIssue merged ke df_issues:')
print('  Issues dengan utterances: {:,}'.format((df_issues['utteranceCountPerIssue'] > 0).sum()))
print('  Issues tanpa utterances : {:,}'.format((df_issues['utteranceCountPerIssue'] == 0).sum()))

Kolom ID di utterances: issueid
Unique issues dengan utterances: 360
count   360.00
mean     40.15
std      39.51
min       6.00
25%      17.00
50%      25.00
75%      48.00
max     359.00
Name: utteranceCountPerIssue, dtype: float64

utteranceCountPerIssue merged ke df_issues:
  Issues dengan utterances: 360
  Issues tanpa utterances : 66,331


## 7. Integrasi Dataset

Bagian ini mengevaluasi apakah DS1 dan DS2 bisa digabungkan secara langsung menggunakan kunci relasi yang sama, atau hanya dianalisis secara terpisah. Keputusan ini penting karena akan memengaruhi cara kita menjawab pertanyaan analitik di tahap berikutnya.

In [ ]:
# Keputusan Integrasi DS1 vs DS2
#
# Kunci evaluasi:
#   DS1 (IBM Watson): simulasi, 100K tiket, tidak ada timestamp, tidak ada PK agent
#   DS2 (Mendeley)  : real helpdesk Jan 2016–Mar 2023, 66K tiket, ada timestamp
#
# Opsi A (concat): TIDAK DIPILIH
#   Schema tidak overlap: DS1 tidak punya wf_*, DS2 tidak punya Severity/Priority numeric
#   Concat akan menghasilkan >90% NaN untuk semua kolom
#
# Opsi B (Analisis Terpisah): DIPILIH untuk DS1 vs DS2
#   Alasan: pertanyaan analitik kelompok bisa dijawab per dataset;
#           DS1 = IBM Watson (simulasi), DS2 = Mendeley (real) — sumber berbeda
#
# Opsi C (Enrichment): DIPILIH untuk enrichment internal DS2
#   utterances -> issues: sudah dilakukan (utteranceCountPerIssue)
#   scored -> issues: akan dilakukan di bawah
#
# ASUMSI MAPPING (jika diperlukan perbandingan lintas dataset):
#   DS1 Severity (1-4) <-> DS2 priorityNormalized (high/medium/low) -- PROXY, bukan equivalent
#   DS1 daysOpen       <-> DS2 resolutionDurationHours              -- PROXY, satuan berbeda
#   DS1 TicketType     <-> DS2 issue_type                          -- PROXY, kategori berbeda
#
# LIMITASI:
#   DS1 tidak bisa dianalisis tren waktu (tidak ada timestamp)
#   DS2 scored sample hanya subset kecil (cek ukuran di 1A)
#   Perbandingan lintas dataset harus dicatat sebagai asumsi di laporan

print('=== Keputusan Integrasi ===')
print()
print('Opsi B: DS1 dan DS2 DIANALISIS TERPISAH')
print('  Tidak ada mergedClean.csv (Opsi A tidak applicable)')
print()
print('Opsi C: Enrichment internal DS2')
print('  utteranceCountPerIssue sudah di-merge ke df_issues ')
print()
print('Asumsi mapping (jika perbandingan lintas dataset diperlukan):')
print('  DS1 Severity <-> DS2 priorityNormalized (PROXY, bukan exact equivalent)')
print('  DS1 daysOpen <-> DS2 resolutionDurationHours (PROXY, satuan berbeda)')

=== Keputusan Integrasi ===

Opsi B: DS1 dan DS2 DIANALISIS TERPISAH
  Tidak ada mergedClean.csv (Opsi A tidak applicable)

Opsi C: Enrichment internal DS2
  utteranceCountPerIssue sudah di-merge ke df_issues 

Asumsi mapping (jika perbandingan lintas dataset diperlukan):
  DS1 Severity <-> DS2 priorityNormalized (PROXY, bukan exact equivalent)
  DS1 daysOpen <-> DS2 resolutionDurationHours (PROXY, satuan berbeda)


In [61]:
# DS2 Internal Enrichment: Link scored ke issues
if q_cols and 'performanceBinary' in df_scored.columns:
    scored_id_col_local = 'id' if 'id' in df_scored.columns else df_scored.columns[0]
    scored_to_merge = df_scored[[scored_id_col_local, 'compositeScore', 'performanceBinary']].copy()
    scored_to_merge = scored_to_merge.rename(columns={scored_id_col_local: 'id'})
    # Drop duplikat id di scored sebelum merge
    scored_to_merge = scored_to_merge.drop_duplicates(subset=['id'])
    
    n_before = len(df_issues)
    df_issues = df_issues.merge(scored_to_merge, on='id', how='left')
    assert len(df_issues) == n_before, 'Merge tidak 1:1 — periksa duplikasi scored!'
    
    n_with_score = df_issues['compositeScore'].notna().sum()
    print('DS2 Enrichment: scored metrics merged ke df_issues')
    print('  Issues dengan compositeScore: {:,} ({:.1f}%)'.format(
        n_with_score, n_with_score/len(df_issues)*100))
    print('  Issues tanpa score           : {:,}'.format(df_issues['compositeScore'].isna().sum()))
else:
    print('Scored sample tidak memiliki Q columns atau performanceBinary — skip enrichment.')

DS2 Enrichment: scored metrics merged ke df_issues
  Issues dengan compositeScore: 360 (0.5%)
  Issues tanpa score           : 66,331


## 8. Menyimpan Hasil Pembersihan

Semua dataset yang sudah bersih dan memiliki fitur-fitur baru disimpan ke folder `data/processed/`. File-file ini akan digunakan sebagai input pada notebook selanjutnya (03_explore.ipynb dan 04_model.ipynb). Setiap file disimpan dalam format CSV agar mudah dibaca oleh semua anggota tim.

In [62]:
# save Output Clean
import os

os.makedirs(OUTPUT_PATH, exist_ok=True)

output_files = {
    'ds1Clean.csv'          : df1,
    'ds2IssuesClean.csv'    : df_issues,
    'ds2UtterancesClean.csv': df_utterances,
    'ds2ScoredClean.csv'    : df_scored,
    'ds2SnapshotClean.csv'  : df_snapshot,
}

print('=== Menyimpan Output ke {} ==='.format(OUTPUT_PATH))
for fname, df in output_files.items():
    fpath = OUTPUT_PATH + fname
    df.to_csv(fpath, index=False)
    size_kb = os.path.getsize(fpath) / 1024
    print('  {:30s}: {:,} baris x {:>3} kolom | {:,.1f} KB'.format(
        fname, df.shape[0], df.shape[1], size_kb))
print()
print('=== Verifikasi Sample ===')
print('DS1 kolom baru:', [c for c in df1.columns if c not in df1_orig.columns])
print('DS2 kolom baru:', [c for c in df_issues.columns if c not in df_issues_orig.columns])
print()
print('DS1 head (kolom terpilih):')
disp_cols_ds1 = [c for c in ['ticket','FiledAgainst','TicketType','severityLevel',
    'priorityLabel','daysOpen','isLongTicket','isHighPriority','resolutionDurationDays'] if c in df1.columns]
display(df1[disp_cols_ds1].head(3))
print()
print('DS2 head (kolom terpilih):')
disp_cols_ds2 = [c for c in ['id','issue_type','priorityNormalized','isResolved',
    'totalTimeHours','resolutionDurationHours','isComplex','resolutionSpeedCategory',
    'utteranceCountPerIssue'] if c in df_issues.columns]
display(df_issues[disp_cols_ds2].head(3))

=== Menyimpan Output ke ../data/processed/ ===


  ds1Clean.csv                  : 100,000 baris x  22 kolom | 13,769.9 KB


  ds2IssuesClean.csv            : 66,691 baris x  57 kolom | 33,793.6 KB
  ds2UtterancesClean.csv        : 14,455 baris x  12 kolom | 3,482.6 KB
  ds2ScoredClean.csv            : 747 baris x  21 kolom | 125.8 KB


  ds2SnapshotClean.csv          : 90,963 baris x  45 kolom | 27,220.9 KB

=== Verifikasi Sample ===
DS1 kolom baru: ['resolutionDurationDays', 'isLongTicket', 'severityLevel', 'severityLabel', 'priorityLevel', 'priorityLabel', 'seniorityLevel', 'seniorityLabel', 'satisfactionLevel', 'satisfactionLabel', 'isHighPriority', 'priorityVerified']
DS2 kolom baru: ['isResolved', 'started_dt', 'ended_dt', 'issue_created_dt', 'issue_resolution_date_dt', 'last_change_date_dt', 'totalTimeHours', 'timePerStepHours', 'resolutionDurationHours', 'isComplex', 'priorityNormalized', 'resolutionSpeedCategory', 'utteranceCountPerIssue', 'compositeScore', 'performanceBinary']

DS1 head (kolom terpilih):


,ticket,FiledAgainst,TicketType,severityLevel,priorityLabel,daysOpen,isLongTicket,isHighPriority,resolutionDurationDays
0,1,systems,issue,2.00,unassigned,3,0,0,3
1,2,software,request,1.00,low,5,0,0,5
2,3,access/login,request,2.00,unassigned,0,0,0,0



DS2 head (kolom terpilih):


,id,issue_type,priorityNormalized,isResolved,totalTimeHours,resolutionDurationHours,isComplex,resolutionSpeedCategory,utteranceCountPerIssue
0,11887.00,ticket,medium,1,0.55,0.55,0,fast,0
1,11890.00,ticket,medium,1,26.40,26.40,0,fast,0
2,11904.00,ticket,medium,1,120.89,120.89,0,fast,0
